# Camada Silver — tratamento e qualidade dos dados

Este notebook realiza a limpeza, padronização, tipagem e validação dos dados provenientes da PREVIC, SUSEP e IBGE.

## Objetivos

- Identificar e renomear as colunas das bases sem cabeçalho.
- Remover espaços desnecessários e padronizar textos.
- Converter períodos, datas, quantidades e valores monetários.
- Tratar valores ausentes e códigos de ausência.
- Transformar as tabelas largas do IBGE em formato analítico.
- Aplicar o recorte temporal até dezembro de 2025.
- Registrar e validar regras de qualidade.
- Gravar tabelas Delta tratadas no schema `workspace.silver`.

## 1. PREVIC — Demonstrativo de Sexo e Idade (DSI)

A base DSI descreve participantes ativos, aposentados e beneficiários de pensão por entidade, sexo e faixa etária.

Na camada Silver:

- as colunas sem cabeçalho recebem nomes descritivos;
- textos são limpos e padronizados;
- códigos e quantidades são convertidos para tipos numéricos;
- o período `AAAAMM` é convertido também em data de referência;
- dois campos constantes, sem informação analítica, são removidos;
- os metadados de origem e ingestão são preservados;
- são mantidos somente dados até dezembro de 2025.

In [0]:
from pyspark.sql import functions as F

# Leitura da tabela Bronze
df_dsi_bronze = spark.table("workspace.bronze.previc_dsi_2025")

# Limpeza, identificação e tipagem das colunas
df_dsi_silver = (
    df_dsi_bronze
    .select(
        F.trim(F.col("_c0")).alias("ano_mes"),
        F.trim(F.col("_c1")).cast("int").alias("codigo_entidade"),
        F.upper(F.trim(F.col("_c2"))).alias("sigla_entidade"),
        F.trim(F.col("_c5")).cast("int").alias("codigo_tipo_populacao"),
        F.trim(F.col("_c6")).cast("int").alias("codigo_conta_populacao"),
        F.trim(F.col("_c7")).alias("descricao_populacao"),
        F.upper(F.trim(F.col("_c8"))).alias("sexo"),
        F.upper(F.trim(F.col("_c9"))).alias("faixa_etaria"),
        F.trim(F.col("_c10")).cast("long").alias("quantidade"),
        F.col("_arquivo_origem"),
        F.col("_data_ingestao")
    )
    .withColumn(
        "data_referencia",
        F.to_date(
            F.concat(F.col("ano_mes"), F.lit("01")),
            "yyyyMMdd"
        )
    )
    .filter(F.col("ano_mes") <= "202512")
)

# Organização final das colunas
df_dsi_silver = df_dsi_silver.select(
    "data_referencia",
    "ano_mes",
    "codigo_entidade",
    "sigla_entidade",
    "codigo_tipo_populacao",
    "codigo_conta_populacao",
    "descricao_populacao",
    "sexo",
    "faixa_etaria",
    "quantidade",
    "_arquivo_origem",
    "_data_ingestao"
)

display(df_dsi_silver.limit(20))

data_referencia,ano_mes,codigo_entidade,sigla_entidade,codigo_tipo_populacao,codigo_conta_populacao,descricao_populacao,sexo,faixa_etaria,quantidade,_arquivo_origem,_data_ingestao
2025-12-01,202512,14,AGROS,19,31000,Participantes Ativos,F,ATÉ 24 ANOS,39,DSI_2025.csv,2026-09-07T02:23:14.537Z
2025-12-01,202512,14,AGROS,19,31000,Participantes Ativos,F,ENTRE 25 E 34 ANOS,114,DSI_2025.csv,2026-09-07T02:23:14.537Z
2025-12-01,202512,14,AGROS,19,31000,Participantes Ativos,F,ENTRE 35 E 54 ANOS,800,DSI_2025.csv,2026-09-07T02:23:14.537Z
2025-12-01,202512,14,AGROS,19,31000,Participantes Ativos,F,ENTRE 55 E 64 ANOS,148,DSI_2025.csv,2026-09-07T02:23:14.537Z
2025-12-01,202512,14,AGROS,19,31000,Participantes Ativos,F,ENTRE 65 E 74 ANOS,88,DSI_2025.csv,2026-09-07T02:23:14.537Z
2025-12-01,202512,14,AGROS,19,31000,Participantes Ativos,F,ENTRE 75 E 84 ANOS,23,DSI_2025.csv,2026-09-07T02:23:14.537Z
2025-12-01,202512,14,AGROS,19,31000,Participantes Ativos,F,MAIOR QUE 85 ANOS,16,DSI_2025.csv,2026-09-07T02:23:14.537Z
2025-12-01,202512,14,AGROS,19,31000,Participantes Ativos,M,ATÉ 24 ANOS,40,DSI_2025.csv,2026-09-07T02:23:14.537Z
2025-12-01,202512,14,AGROS,19,31000,Participantes Ativos,M,ENTRE 25 E 34 ANOS,126,DSI_2025.csv,2026-09-07T02:23:14.537Z
2025-12-01,202512,14,AGROS,19,31000,Participantes Ativos,M,ENTRE 35 E 54 ANOS,905,DSI_2025.csv,2026-09-07T02:23:14.537Z


In [0]:
# Validações de qualidade da base DSI

df_validacao_dsi = df_dsi_silver.agg(
    F.count("*").alias("quantidade_registros"),
    F.countDistinct("codigo_entidade").alias("quantidade_entidades"),
    F.sum(
        F.when(F.col("quantidade").isNull(), 1).otherwise(0)
    ).alias("quantidades_nulas"),
    F.sum(
        F.when(F.col("quantidade") < 0, 1).otherwise(0)
    ).alias("quantidades_negativas"),
    F.sum(
        F.when(F.col("data_referencia").isNull(), 1).otherwise(0)
    ).alias("datas_invalidas")
)

display(df_validacao_dsi)

quantidade_registros,quantidade_entidades,quantidades_nulas,quantidades_negativas,datas_invalidas
9938,242,0,12,0


In [0]:
# Investigação das quantidades negativas encontradas

df_dsi_negativos = (
    df_dsi_silver
    .filter(F.col("quantidade") < 0)
    .select(
        "codigo_entidade",
        "sigla_entidade",
        "codigo_tipo_populacao",
        "descricao_populacao",
        "sexo",
        "faixa_etaria",
        "quantidade",
        "_arquivo_origem"
    )
    .orderBy(
        "codigo_entidade",
        "descricao_populacao",
        "sexo",
        "faixa_etaria"
    )
)

display(df_dsi_negativos)

codigo_entidade,sigla_entidade,codigo_tipo_populacao,descricao_populacao,sexo,faixa_etaria,quantidade,_arquivo_origem
1198,CASFAM,23,Assistidos - Aposentados,F,ENTRE 75 E 84 ANOS,-8,DSI_2025.csv
1198,CASFAM,23,Assistidos - Aposentados,F,MAIOR QUE 85 ANOS,-18,DSI_2025.csv
1198,CASFAM,23,Assistidos - Aposentados,M,ENTRE 75 E 84 ANOS,-8,DSI_2025.csv
1198,CASFAM,23,Assistidos - Aposentados,M,MAIOR QUE 85 ANOS,-3,DSI_2025.csv
1198,CASFAM,19,Participantes Ativos,F,ENTRE 75 E 84 ANOS,-6,DSI_2025.csv
1198,CASFAM,19,Participantes Ativos,F,MAIOR QUE 85 ANOS,-15,DSI_2025.csv
1198,CASFAM,19,Participantes Ativos,M,ENTRE 75 E 84 ANOS,-10,DSI_2025.csv
1198,CASFAM,19,Participantes Ativos,M,MAIOR QUE 85 ANOS,-9,DSI_2025.csv
1359,E-INVEST,23,Assistidos - Aposentados,M,MAIOR QUE 85 ANOS,-1,DSI_2025.csv
1359,E-INVEST,19,Participantes Ativos,M,ENTRE 25 E 34 ANOS,-20,DSI_2025.csv


### Tratamento de quantidades negativas

Foram identificados 12 registros com quantidades negativas, concentrados nas entidades CASFAM e E-INVEST.

Como a variável representa uma quantidade de pessoas, valores negativos não possuem interpretação válida. Para preservar a rastreabilidade, o valor original foi mantido em `quantidade_informada`, enquanto o campo analítico `quantidade` recebeu valor nulo. Também foi criada uma flag de qualidade para identificar esses registros.

Os valores não foram substituídos por zero, pois zero indicaria ausência de participantes, informação diferente de um dado inválido.

In [0]:
# Preservação do valor original e tratamento das quantidades negativas

df_dsi_silver = (
    df_dsi_silver
    .withColumnRenamed("quantidade", "quantidade_informada")
    .withColumn(
        "flag_quantidade_invalida",
        F.col("quantidade_informada") < 0
    )
    .withColumn(
        "quantidade",
        F.when(
            F.col("quantidade_informada") >= 0,
            F.col("quantidade_informada")
        ).otherwise(F.lit(None).cast("long"))
    )
    .select(
        "data_referencia",
        "ano_mes",
        "codigo_entidade",
        "sigla_entidade",
        "codigo_tipo_populacao",
        "codigo_conta_populacao",
        "descricao_populacao",
        "sexo",
        "faixa_etaria",
        "quantidade",
        "quantidade_informada",
        "flag_quantidade_invalida",
        "_arquivo_origem",
        "_data_ingestao"
    )
)

display(
    df_dsi_silver
    .filter(F.col("flag_quantidade_invalida"))
)

data_referencia,ano_mes,codigo_entidade,sigla_entidade,codigo_tipo_populacao,codigo_conta_populacao,descricao_populacao,sexo,faixa_etaria,quantidade,quantidade_informada,flag_quantidade_invalida,_arquivo_origem,_data_ingestao
2025-12-01,202512,1198,CASFAM,19,31000,Participantes Ativos,F,ENTRE 75 E 84 ANOS,null,-6,true,DSI_2025.csv,2026-09-07T02:23:14.537Z
2025-12-01,202512,1198,CASFAM,19,31000,Participantes Ativos,F,MAIOR QUE 85 ANOS,null,-15,true,DSI_2025.csv,2026-09-07T02:23:14.537Z
2025-12-01,202512,1198,CASFAM,19,31000,Participantes Ativos,M,ENTRE 75 E 84 ANOS,null,-10,true,DSI_2025.csv,2026-09-07T02:23:14.537Z
2025-12-01,202512,1198,CASFAM,19,31000,Participantes Ativos,M,MAIOR QUE 85 ANOS,null,-9,true,DSI_2025.csv,2026-09-07T02:23:14.537Z
2025-12-01,202512,1198,CASFAM,23,32000,Assistidos - Aposentados,F,ENTRE 75 E 84 ANOS,null,-8,true,DSI_2025.csv,2026-09-07T02:23:14.537Z
2025-12-01,202512,1198,CASFAM,23,32000,Assistidos - Aposentados,F,MAIOR QUE 85 ANOS,null,-18,true,DSI_2025.csv,2026-09-07T02:23:14.537Z
2025-12-01,202512,1198,CASFAM,23,32000,Assistidos - Aposentados,M,ENTRE 75 E 84 ANOS,null,-8,true,DSI_2025.csv,2026-09-07T02:23:14.537Z
2025-12-01,202512,1198,CASFAM,23,32000,Assistidos - Aposentados,M,MAIOR QUE 85 ANOS,null,-3,true,DSI_2025.csv,2026-09-07T02:23:14.537Z
2025-12-01,202512,1359,E-INVEST,19,31000,Participantes Ativos,M,ENTRE 25 E 34 ANOS,null,-20,true,DSI_2025.csv,2026-09-07T02:23:14.537Z
2025-12-01,202512,1359,E-INVEST,19,31000,Participantes Ativos,M,ENTRE 75 E 84 ANOS,null,-5,true,DSI_2025.csv,2026-09-07T02:23:14.537Z


In [0]:
# Gravação da tabela DSI tratada na camada Silver

(
    df_dsi_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.previc_dsi_2025")
)

print("Tabela workspace.silver.previc_dsi_2025 gravada com sucesso.")

Tabela workspace.silver.previc_dsi_2025 gravada com sucesso.


In [0]:
# Validação após a gravação em Delta

df_dsi_gravado = spark.table("workspace.silver.previc_dsi_2025")

df_validacao_final_dsi = df_dsi_gravado.agg(
    F.count("*").alias("quantidade_registros"),
    F.countDistinct("codigo_entidade").alias("quantidade_entidades"),
    F.sum(
        F.when(F.col("flag_quantidade_invalida"), 1).otherwise(0)
    ).alias("registros_invalidos_sinalizados"),
    F.sum(
        F.when(F.col("quantidade").isNull(), 1).otherwise(0)
    ).alias("quantidades_nulas_apos_tratamento"),
    F.sum(
        F.when(F.col("quantidade") < 0, 1).otherwise(0)
    ).alias("quantidades_negativas_na_coluna_analitica")
)

display(df_validacao_final_dsi)

quantidade_registros,quantidade_entidades,registros_invalidos_sinalizados,quantidades_nulas_apos_tratamento,quantidades_negativas_na_coluna_analitica
9938,242,12,12,0


## 2. PREVIC — Estatística de Planos de Benefícios (EPB)

Os arquivos EPB contêm informações estatísticas dos planos de benefícios das entidades fechadas de previdência complementar.

Foram disponibilizados dois arquivos referentes ao primeiro e ao segundo semestre de 2025. Antes da consolidação, serão verificadas a estrutura, a correspondência das colunas e a qualidade dos dados de cada período.

In [0]:
# Leitura das tabelas EPB da camada Bronze

df_epb_1sem_bronze = spark.table(
    "workspace.bronze.previc_epb_1semestre_2025"
)

df_epb_2sem_bronze = spark.table(
    "workspace.bronze.previc_epb_2semestre_2025"
)

print("Schema do EPB — 1º semestre:")
df_epb_1sem_bronze.printSchema()

print("Schema do EPB — 2º semestre:")
df_epb_2sem_bronze.printSchema()

Schema do EPB — 1º semestre:
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)
 |-- _c12: string (nullable = true)
 |-- _c13: string (nullable = true)
 |-- _c14: string (nullable = true)
 |-- _c15: string (nullable = true)
 |-- _arquivo_origem: string (nullable = true)
 |-- _data_ingestao: timestamp (nullable = true)

Schema do EPB — 2º semestre:
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullab

In [0]:
# Amostra do primeiro semestre

display(df_epb_1sem_bronze.limit(20))

_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9,_c10,_c11,_c12,_c13,_c14,_c15,_arquivo_origem,_data_ingestao
202501,941,SERPROS,EFPC,NULL,0,7,12000,Auxílios - Prestação Continuada,FOLHA,17,4,6,6,15,2026-06-26,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
202501,941,SERPROS,EFPC,NULL,0,21,31200,Participante - com custeio patronal e do participante,FOLHA,7466,11,129,129,7348,2026-06-26,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
202501,941,SERPROS,EFPC,NULL,0,23,32000,Assistidos - Aposentados,FOLHA,4784,38,1,1,4821,2026-06-26,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
202501,941,SERPROS,EFPC,NULL,0,24,33000,Assistidos - Beneficiários de Pensão,FOLHA,1031,4,4,4,1031,2026-06-26,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
202501,941,SERPROS,EFPC,NULL,0,25,34000,Designados,FOLHA,24953,16,3,3,24966,2026-06-26,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
202502,941,SERPROS,EFPC,NULL,0,7,12000,Auxílios - Prestação Continuada,FOLHA,15,5,6,6,14,2026-06-26,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
202502,941,SERPROS,EFPC,NULL,0,21,31200,Participante - com custeio patronal e do participante,FOLHA,7348,35,90,90,7293,2026-06-26,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
202502,941,SERPROS,EFPC,NULL,0,23,32000,Assistidos - Aposentados,FOLHA,4821,44,6,6,4859,2026-06-26,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
202502,941,SERPROS,EFPC,NULL,0,24,33000,Assistidos - Beneficiários de Pensão,FOLHA,1031,7,3,3,1035,2026-06-26,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
202502,941,SERPROS,EFPC,NULL,0,25,34000,Designados,FOLHA,24966,53,4,4,25015,2026-06-26,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z


In [0]:
# Amostra do segundo semestre

display(df_epb_2sem_bronze.limit(20))

_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9,_c10,_c11,_c12,_c13,_c14,_c15,_arquivo_origem,_data_ingestao
202507,3558,PREVICEL,EFPC,NULL,0,7,12000,Auxílios - Prestação Continuada,FOLHA,0,0,0,0,0,2026-06-26,EPB_2SEMESTRE_2025.csv,2026-09-07T02:23:26.816Z
202507,3558,PREVICEL,EFPC,NULL,0,21,31200,Participante - com custeio patronal e do participante,FOLHA,819,1,4,4,816,2026-06-26,EPB_2SEMESTRE_2025.csv,2026-09-07T02:23:26.816Z
202507,3558,PREVICEL,EFPC,NULL,0,23,32000,Assistidos - Aposentados,FOLHA,181,0,0,0,181,2026-06-26,EPB_2SEMESTRE_2025.csv,2026-09-07T02:23:26.816Z
202507,3558,PREVICEL,EFPC,NULL,0,24,33000,Assistidos - Beneficiários de Pensão,FOLHA,47,0,0,0,47,2026-06-26,EPB_2SEMESTRE_2025.csv,2026-09-07T02:23:26.816Z
202507,3558,PREVICEL,EFPC,NULL,0,25,34000,Designados,FOLHA,1121,2,9,9,1114,2026-06-26,EPB_2SEMESTRE_2025.csv,2026-09-07T02:23:26.816Z
202508,3558,PREVICEL,EFPC,NULL,0,7,12000,Auxílios - Prestação Continuada,FOLHA,0,0,0,0,0,2026-06-26,EPB_2SEMESTRE_2025.csv,2026-09-07T02:23:26.816Z
202508,3558,PREVICEL,EFPC,NULL,0,21,31200,Participante - com custeio patronal e do participante,FOLHA,816,1,6,6,811,2026-06-26,EPB_2SEMESTRE_2025.csv,2026-09-07T02:23:26.816Z
202508,3558,PREVICEL,EFPC,NULL,0,23,32000,Assistidos - Aposentados,FOLHA,181,2,0,0,183,2026-06-26,EPB_2SEMESTRE_2025.csv,2026-09-07T02:23:26.816Z
202508,3558,PREVICEL,EFPC,NULL,0,24,33000,Assistidos - Beneficiários de Pensão,FOLHA,47,0,0,0,47,2026-06-26,EPB_2SEMESTRE_2025.csv,2026-09-07T02:23:26.816Z
202508,3558,PREVICEL,EFPC,NULL,0,25,34000,Designados,FOLHA,1114,2,14,14,1102,2026-06-26,EPB_2SEMESTRE_2025.csv,2026-09-07T02:23:26.816Z


In [0]:
# Visualização dos campos numéricos finais do EPB

display(
    df_epb_1sem_bronze
    .select(
        "_c0",
        "_c1",
        "_c2",
        "_c6",
        "_c7",
        "_c8",
        "_c9",
        "_c10",
        "_c11",
        "_c12",
        "_c13",
        "_c14",
        "_c15"
    )
    .limit(30)
)

_c0,_c1,_c2,_c6,_c7,_c8,_c9,_c10,_c11,_c12,_c13,_c14,_c15
202501,941,SERPROS,7,12000,Auxílios - Prestação Continuada,FOLHA,17,4,6,6,15,2026-06-26
202501,941,SERPROS,21,31200,Participante - com custeio patronal e do participante,FOLHA,7466,11,129,129,7348,2026-06-26
202501,941,SERPROS,23,32000,Assistidos - Aposentados,FOLHA,4784,38,1,1,4821,2026-06-26
202501,941,SERPROS,24,33000,Assistidos - Beneficiários de Pensão,FOLHA,1031,4,4,4,1031,2026-06-26
202501,941,SERPROS,25,34000,Designados,FOLHA,24953,16,3,3,24966,2026-06-26
202502,941,SERPROS,7,12000,Auxílios - Prestação Continuada,FOLHA,15,5,6,6,14,2026-06-26
202502,941,SERPROS,21,31200,Participante - com custeio patronal e do participante,FOLHA,7348,35,90,90,7293,2026-06-26
202502,941,SERPROS,23,32000,Assistidos - Aposentados,FOLHA,4821,44,6,6,4859,2026-06-26
202502,941,SERPROS,24,33000,Assistidos - Beneficiários de Pensão,FOLHA,1031,7,3,3,1035,2026-06-26
202502,941,SERPROS,25,34000,Designados,FOLHA,24966,53,4,4,25015,2026-06-26


In [0]:
# Perfil dos campos finais do EPB

for coluna in ["_c9", "_c12", "_c13", "_c14", "_c15"]:
    print(f"Valores mais frequentes de {coluna}:")
    
    display(
        df_epb_1sem_bronze
        .groupBy(coluna)
        .count()
        .orderBy(F.desc("count"))
        .limit(20)
    )

Valores mais frequentes de _c9:


_c9,count
FOLHA,150424
TOTALIZADOR,15801


Valores mais frequentes de _c12:


_c12,count
0,130530
1,9046
2,4967
3,3023
4,2075
5,1524
6,1246
7,977
8,883
9,806


Valores mais frequentes de _c13:


_c13,count
0,130530
1,9046
2,4967
3,3023
4,2075
5,1524
6,1246
7,977
8,883
9,806


Valores mais frequentes de _c14:


_c14,count
0,78036
1,6593
2,3891
3,2809
4,2349
5,2055
6,1707
7,1462
8,1206
9,1166


Valores mais frequentes de _c15:


_c15,count
2026-06-26,166225


In [0]:
# Verificação da relação entre os campos numéricos do EPB

def validar_campos_epb(df, semestre):
    return (
        df
        .select(
            F.lit(semestre).alias("semestre"),
            F.col("_c10").cast("long").alias("quantidade_inicial"),
            F.col("_c11").cast("long").alias("entradas"),
            F.col("_c12").cast("long").alias("campo_c12"),
            F.col("_c13").cast("long").alias("saidas"),
            F.col("_c14").cast("long").alias("quantidade_final")
        )
        .agg(
            F.first("semestre").alias("semestre"),
            F.count("*").alias("registros"),
            F.sum(
                F.when(
                    F.col("campo_c12") != F.col("saidas"), 1
                ).otherwise(0)
            ).alias("diferencas_c12_c13"),
            F.sum(
                F.when(
                    F.col("quantidade_inicial")
                    + F.col("entradas")
                    - F.col("saidas")
                    != F.col("quantidade_final"),
                    1
                ).otherwise(0)
            ).alias("balancos_inconsistentes"),
            F.sum(
                F.when(
                    (F.col("quantidade_inicial") < 0)
                    | (F.col("entradas") < 0)
                    | (F.col("campo_c12") < 0)
                    | (F.col("saidas") < 0)
                    | (F.col("quantidade_final") < 0),
                    1
                ).otherwise(0)
            ).alias("registros_com_negativos")
        )
    )


df_validacao_campos_epb = (
    validar_campos_epb(df_epb_1sem_bronze, "1º semestre")
    .unionByName(
        validar_campos_epb(df_epb_2sem_bronze, "2º semestre")
    )
)

display(df_validacao_campos_epb)

semestre,registros,diferencas_c12_c13,balancos_inconsistentes,registros_com_negativos
1º semestre,166225,0,0,510
2º semestre,165661,0,0,123


In [0]:
# Localização dos valores negativos por campo

def resumir_negativos_epb(df, semestre):
    return df.agg(
        F.count("*").alias("registros"),
        
        F.sum(
            F.when(F.col("_c10").cast("long") < 0, 1).otherwise(0)
        ).alias("negativos_quantidade_inicial"),
        
        F.sum(
            F.when(F.col("_c11").cast("long") < 0, 1).otherwise(0)
        ).alias("negativos_entradas"),
        
        F.sum(
            F.when(F.col("_c12").cast("long") < 0, 1).otherwise(0)
        ).alias("negativos_c12"),
        
        F.sum(
            F.when(F.col("_c13").cast("long") < 0, 1).otherwise(0)
        ).alias("negativos_saidas"),
        
        F.sum(
            F.when(F.col("_c14").cast("long") < 0, 1).otherwise(0)
        ).alias("negativos_quantidade_final"),
        
        F.min(F.col("_c10").cast("long")).alias("minimo_inicial"),
        F.min(F.col("_c11").cast("long")).alias("minimo_entradas"),
        F.min(F.col("_c13").cast("long")).alias("minimo_saidas"),
        F.min(F.col("_c14").cast("long")).alias("minimo_final")
    ).withColumn("semestre", F.lit(semestre))


df_resumo_negativos_epb = (
    resumir_negativos_epb(df_epb_1sem_bronze, "1º semestre")
    .unionByName(
        resumir_negativos_epb(df_epb_2sem_bronze, "2º semestre")
    )
    .select(
        "semestre",
        "registros",
        "negativos_quantidade_inicial",
        "negativos_entradas",
        "negativos_c12",
        "negativos_saidas",
        "negativos_quantidade_final",
        "minimo_inicial",
        "minimo_entradas",
        "minimo_saidas",
        "minimo_final"
    )
)

display(df_resumo_negativos_epb)

semestre,registros,negativos_quantidade_inicial,negativos_entradas,negativos_c12,negativos_saidas,negativos_quantidade_final,minimo_inicial,minimo_entradas,minimo_saidas,minimo_final
1º semestre,166225,0,2,0,0,508,0,-7,0,-21
2º semestre,165661,0,0,0,0,123,0,0,0,-2071


In [0]:
# Amostra dos registros EPB com valores negativos

display(
    df_epb_1sem_bronze
    .filter(
        (F.col("_c10").cast("long") < 0)
        | (F.col("_c11").cast("long") < 0)
        | (F.col("_c13").cast("long") < 0)
        | (F.col("_c14").cast("long") < 0)
    )
    .select(
        "_c0", "_c1", "_c2", "_c6", "_c8", "_c9",
        "_c10", "_c11", "_c12", "_c13", "_c14", "_c15"
    )
    .limit(30)
)

_c0,_c1,_c2,_c6,_c8,_c9,_c10,_c11,_c12,_c13,_c14,_c15
202502,941,SERPROS,17,Portabilidade - Plano de Benefícios Originário,FOLHA,0,0,5,5,-5,2026-06-26
202502,941,SERPROS,17,Portabilidade - Plano de Benefícios Originário,FOLHA,0,0,5,5,-5,2026-06-26
202501,1239,FUNCESP,17,Portabilidade - Plano de Benefícios Originário,FOLHA,0,0,5,5,-5,2026-06-26
202501,1972,PROMON,17,Portabilidade - Plano de Benefícios Originário,FOLHA,0,0,3,3,-3,2026-06-26
202501,1972,PROMON,16,Portabilidade (totalizador),FOLHA,0,1,3,3,-2,2026-06-26
202501,1239,FUNCESP,17,Portabilidade - Plano de Benefícios Originário,FOLHA,0,0,5,5,-5,2026-06-26
202501,1239,FUNCESP,16,Portabilidade (totalizador),FOLHA,0,0,5,5,-5,2026-06-26
202501,1972,PROMON,17,Portabilidade - Plano de Benefícios Originário,FOLHA,0,0,3,3,-3,2026-06-26
202501,1972,PROMON,16,Portabilidade (totalizador),FOLHA,0,1,3,3,-2,2026-06-26
202502,223,CENTRUS,17,Portabilidade - Plano de Benefícios Originário,FOLHA,0,0,1,1,-1,2026-06-26


### Tratamento e consolidação dos arquivos EPB

Os arquivos dos dois semestres possuem a mesma estrutura e períodos complementares, permitindo sua consolidação.

Foi verificado que os campos `_c12` e `_c13` são idênticos em todos os registros. Por isso, somente um deles foi mantido como quantidade de saídas, evitando redundância.

A equação entre quantidade inicial, entradas, saídas e quantidade final apresentou consistência em todos os registros. Entretanto, foram encontradas entradas e quantidades finais negativas.

Como esses campos representam quantidades, os valores negativos foram preservados nos campos com sufixo `_informada`, mas transformados em nulos nos campos analíticos. Flags de qualidade foram adicionadas para permitir sua identificação.

In [0]:
# Função de tratamento dos arquivos EPB

def preparar_epb_silver(df, semestre_origem):
    return (
        df
        .select(
            F.trim(F.col("_c0")).alias("ano_mes"),
            F.trim(F.col("_c1")).cast("int").alias("codigo_entidade"),
            F.upper(F.trim(F.col("_c2"))).alias("sigla_entidade"),
            F.upper(F.trim(F.col("_c3"))).alias("tipo_entidade"),
            F.trim(F.col("_c6")).cast("int").alias("codigo_tipo_informacao"),
            F.trim(F.col("_c7")).cast("int").alias("codigo_conta"),
            F.trim(F.col("_c8")).alias("descricao_conta"),
            F.upper(F.trim(F.col("_c9"))).alias("origem_informacao"),
            F.trim(F.col("_c10")).cast("long")
                .alias("quantidade_inicial_informada"),
            F.trim(F.col("_c11")).cast("long")
                .alias("entradas_informadas"),
            F.trim(F.col("_c13")).cast("long")
                .alias("saidas_informadas"),
            F.trim(F.col("_c14")).cast("long")
                .alias("quantidade_final_informada"),
            F.to_date(F.trim(F.col("_c15")), "yyyy-MM-dd")
                .alias("data_atualizacao_fonte"),
            F.lit(semestre_origem).alias("semestre_origem"),
            F.col("_arquivo_origem"),
            F.col("_data_ingestao")
        )
        .withColumn(
            "data_referencia",
            F.to_date(
                F.concat(F.col("ano_mes"), F.lit("01")),
                "yyyyMMdd"
            )
        )
        .filter(F.col("ano_mes") <= "202512")
        .withColumn(
            "flag_entrada_invalida",
            F.col("entradas_informadas") < 0
        )
        .withColumn(
            "flag_quantidade_final_invalida",
            F.col("quantidade_final_informada") < 0
        )
        .withColumn(
            "flag_registro_invalido",
            F.col("flag_entrada_invalida")
            | F.col("flag_quantidade_final_invalida")
        )
        .withColumn(
            "quantidade_inicial",
            F.when(
                F.col("quantidade_inicial_informada") >= 0,
                F.col("quantidade_inicial_informada")
            ).otherwise(F.lit(None).cast("long"))
        )
        .withColumn(
            "entradas",
            F.when(
                F.col("entradas_informadas") >= 0,
                F.col("entradas_informadas")
            ).otherwise(F.lit(None).cast("long"))
        )
        .withColumn(
            "saidas",
            F.when(
                F.col("saidas_informadas") >= 0,
                F.col("saidas_informadas")
            ).otherwise(F.lit(None).cast("long"))
        )
        .withColumn(
            "quantidade_final",
            F.when(
                F.col("quantidade_final_informada") >= 0,
                F.col("quantidade_final_informada")
            ).otherwise(F.lit(None).cast("long"))
        )
    )


# Tratamento individual e consolidação dos semestres

df_epb_1sem_silver = preparar_epb_silver(
    df_epb_1sem_bronze,
    "1º semestre"
)

df_epb_2sem_silver = preparar_epb_silver(
    df_epb_2sem_bronze,
    "2º semestre"
)

df_epb_silver = (
    df_epb_1sem_silver
    .unionByName(df_epb_2sem_silver)
    .select(
        "data_referencia",
        "ano_mes",
        "codigo_entidade",
        "sigla_entidade",
        "tipo_entidade",
        "codigo_tipo_informacao",
        "codigo_conta",
        "descricao_conta",
        "origem_informacao",
        "quantidade_inicial",
        "entradas",
        "saidas",
        "quantidade_final",
        "quantidade_inicial_informada",
        "entradas_informadas",
        "saidas_informadas",
        "quantidade_final_informada",
        "flag_entrada_invalida",
        "flag_quantidade_final_invalida",
        "flag_registro_invalido",
        "data_atualizacao_fonte",
        "semestre_origem",
        "_arquivo_origem",
        "_data_ingestao"
    )
)

display(df_epb_silver.limit(20))

data_referencia,ano_mes,codigo_entidade,sigla_entidade,tipo_entidade,codigo_tipo_informacao,codigo_conta,descricao_conta,origem_informacao,quantidade_inicial,entradas,saidas,quantidade_final,quantidade_inicial_informada,entradas_informadas,saidas_informadas,quantidade_final_informada,flag_entrada_invalida,flag_quantidade_final_invalida,flag_registro_invalido,data_atualizacao_fonte,semestre_origem,_arquivo_origem,_data_ingestao
2025-01-01,202501,941,SERPROS,EFPC,7,12000,Auxílios - Prestação Continuada,FOLHA,17,4,6,15,17,4,6,15,false,false,false,2026-06-26,1º semestre,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
2025-01-01,202501,941,SERPROS,EFPC,21,31200,Participante - com custeio patronal e do participante,FOLHA,7466,11,129,7348,7466,11,129,7348,false,false,false,2026-06-26,1º semestre,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
2025-01-01,202501,941,SERPROS,EFPC,23,32000,Assistidos - Aposentados,FOLHA,4784,38,1,4821,4784,38,1,4821,false,false,false,2026-06-26,1º semestre,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
2025-01-01,202501,941,SERPROS,EFPC,24,33000,Assistidos - Beneficiários de Pensão,FOLHA,1031,4,4,1031,1031,4,4,1031,false,false,false,2026-06-26,1º semestre,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
2025-01-01,202501,941,SERPROS,EFPC,25,34000,Designados,FOLHA,24953,16,3,24966,24953,16,3,24966,false,false,false,2026-06-26,1º semestre,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
2025-02-01,202502,941,SERPROS,EFPC,7,12000,Auxílios - Prestação Continuada,FOLHA,15,5,6,14,15,5,6,14,false,false,false,2026-06-26,1º semestre,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
2025-02-01,202502,941,SERPROS,EFPC,21,31200,Participante - com custeio patronal e do participante,FOLHA,7348,35,90,7293,7348,35,90,7293,false,false,false,2026-06-26,1º semestre,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
2025-02-01,202502,941,SERPROS,EFPC,23,32000,Assistidos - Aposentados,FOLHA,4821,44,6,4859,4821,44,6,4859,false,false,false,2026-06-26,1º semestre,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
2025-02-01,202502,941,SERPROS,EFPC,24,33000,Assistidos - Beneficiários de Pensão,FOLHA,1031,7,3,1035,1031,7,3,1035,false,false,false,2026-06-26,1º semestre,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
2025-02-01,202502,941,SERPROS,EFPC,25,34000,Designados,FOLHA,24966,53,4,25015,24966,53,4,25015,false,false,false,2026-06-26,1º semestre,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z


In [0]:
display(
    df_epb_silver
    .filter(F.col("flag_registro_invalido"))
    .select(
        "ano_mes",
        "sigla_entidade",
        "descricao_conta",
        "quantidade_inicial",
        "entradas",
        "saidas",
        "quantidade_final",
        "entradas_informadas",
        "quantidade_final_informada",
        "flag_entrada_invalida",
        "flag_quantidade_final_invalida"
    )
    .limit(30)
)

ano_mes,sigla_entidade,descricao_conta,quantidade_inicial,entradas,saidas,quantidade_final,entradas_informadas,quantidade_final_informada,flag_entrada_invalida,flag_quantidade_final_invalida
202502,SERPROS,Portabilidade - Plano de Benefícios Originário,0,0,5,null,0,-5,false,true
202502,SERPROS,Portabilidade - Plano de Benefícios Originário,0,0,5,null,0,-5,false,true
202501,FUNCESP,Portabilidade - Plano de Benefícios Originário,0,0,5,null,0,-5,false,true
202501,PROMON,Portabilidade - Plano de Benefícios Originário,0,0,3,null,0,-3,false,true
202501,PROMON,Portabilidade (totalizador),0,1,3,null,1,-2,false,true
202501,FUNCESP,Portabilidade - Plano de Benefícios Originário,0,0,5,null,0,-5,false,true
202501,FUNCESP,Portabilidade (totalizador),0,0,5,null,0,-5,false,true
202501,PROMON,Portabilidade - Plano de Benefícios Originário,0,0,3,null,0,-3,false,true
202501,PROMON,Portabilidade (totalizador),0,1,3,null,1,-2,false,true
202502,CENTRUS,Portabilidade - Plano de Benefícios Originário,0,0,1,null,0,-1,false,true


In [0]:
# Validação da base EPB consolidada

df_validacao_epb = (
    df_epb_silver
    .groupBy("semestre_origem")
    .agg(
        F.count("*").alias("quantidade_registros"),
        F.min("ano_mes").alias("primeiro_periodo"),
        F.max("ano_mes").alias("ultimo_periodo"),
        F.countDistinct("codigo_entidade")
            .alias("quantidade_entidades"),
        F.sum(
            F.when(F.col("flag_entrada_invalida"), 1).otherwise(0)
        ).alias("entradas_invalidas"),
        F.sum(
            F.when(
                F.col("flag_quantidade_final_invalida"), 1
            ).otherwise(0)
        ).alias("quantidades_finais_invalidas"),
        F.sum(
            F.when(F.col("flag_registro_invalido"), 1).otherwise(0)
        ).alias("registros_sinalizados"),
        F.sum(
            F.when(F.col("data_referencia").isNull(), 1).otherwise(0)
        ).alias("datas_referencia_invalidas"),
        F.sum(
            F.when(
                F.col("data_atualizacao_fonte").isNull(), 1
            ).otherwise(0)
        ).alias("datas_atualizacao_invalidas")
    )
    .orderBy("semestre_origem")
)

display(df_validacao_epb)

semestre_origem,quantidade_registros,primeiro_periodo,ultimo_periodo,quantidade_entidades,entradas_invalidas,quantidades_finais_invalidas,registros_sinalizados,datas_referencia_invalidas,datas_atualizacao_invalidas
1º semestre,166225,202501,202506,246,2,508,510,0,0
2º semestre,165661,202507,202512,244,0,123,123,0,0


In [0]:
# Conferência do total consolidado e de duplicidades

colunas_chave_epb = [
    "ano_mes",
    "codigo_entidade",
    "codigo_tipo_informacao",
    "codigo_conta",
    "origem_informacao"
]

quantidade_total_epb = df_epb_silver.count()

quantidade_duplicidades_epb = (
    df_epb_silver
    .groupBy(*colunas_chave_epb)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Quantidade total consolidada: {quantidade_total_epb}")
print(f"Combinações duplicadas na chave proposta: {quantidade_duplicidades_epb}")

Quantidade total consolidada: 331886
Combinações duplicadas na chave proposta: 61392


In [0]:
# Verificação de linhas exatamente iguais no conteúdo de origem

colunas_conteudo_epb = [
    "ano_mes",
    "codigo_entidade",
    "sigla_entidade",
    "tipo_entidade",
    "codigo_tipo_informacao",
    "codigo_conta",
    "descricao_conta",
    "origem_informacao",
    "quantidade_inicial_informada",
    "entradas_informadas",
    "saidas_informadas",
    "quantidade_final_informada",
    "data_atualizacao_fonte"
]

total_registros_epb = df_epb_silver.count()

total_linhas_distintas_epb = (
    df_epb_silver
    .select(*colunas_conteudo_epb)
    .distinct()
    .count()
)

linhas_excedentes_identicas = (
    total_registros_epb - total_linhas_distintas_epb
)

print(f"Total de registros: {total_registros_epb}")
print(f"Linhas distintas pelo conteúdo: {total_linhas_distintas_epb}")
print(f"Linhas excedentes exatamente iguais: {linhas_excedentes_identicas}")

Total de registros: 331886
Linhas distintas pelo conteúdo: 223835
Linhas excedentes exatamente iguais: 108051


In [0]:
# Amostra das combinações repetidas na chave proposta

df_chaves_repetidas_epb = (
    df_epb_silver
    .groupBy(*colunas_chave_epb)
    .agg(
        F.count("*").alias("quantidade_ocorrencias"),
        F.countDistinct("descricao_conta")
            .alias("descricoes_distintas"),
        F.countDistinct("quantidade_inicial_informada")
            .alias("quantidades_iniciais_distintas"),
        F.countDistinct("quantidade_final_informada")
            .alias("quantidades_finais_distintas")
    )
    .filter(F.col("quantidade_ocorrencias") > 1)
    .orderBy(F.desc("quantidade_ocorrencias"))
)

display(df_chaves_repetidas_epb.limit(30))

ano_mes,codigo_entidade,codigo_tipo_informacao,codigo_conta,origem_informacao,quantidade_ocorrencias,descricoes_distintas,quantidades_iniciais_distintas,quantidades_finais_distintas
202504,2258,25,34000,FOLHA,96,1,36,36
202506,2258,16,24000,FOLHA,96,1,2,2
202501,2258,21,31200,FOLHA,96,1,86,87
202504,2258,23,32000,FOLHA,96,1,55,52
202501,2258,15,23000,FOLHA,96,1,1,1
202502,2258,9,14000,FOLHA,96,1,21,21
202501,2258,12,17000,FOLHA,96,1,2,2
202501,2258,23,32000,FOLHA,96,1,46,51
202503,2258,25,34000,FOLHA,96,1,36,36
202504,2258,24,33000,FOLHA,96,1,20,20


In [0]:
# Verificação dos campos _c4 e _c5 antes de descartá-los

for nome, df in [
    ("1º semestre", df_epb_1sem_bronze),
    ("2º semestre", df_epb_2sem_bronze)
]:
    print(f"Perfil de _c4 e _c5 — {nome}")

    display(
        df.agg(
            F.countDistinct("_c4").alias("valores_distintos_c4"),
            F.countDistinct("_c5").alias("valores_distintos_c5"),
            F.sum(
                F.when(
                    F.col("_c4").isNotNull()
                    & (F.upper(F.trim(F.col("_c4"))) != "NULL"),
                    1
                ).otherwise(0)
            ).alias("registros_com_informacao_c4"),
            F.sum(
                F.when(
                    F.trim(F.col("_c5")) != "0",
                    1
                ).otherwise(0)
            ).alias("registros_c5_diferente_zero")
        )
    )

Perfil de _c4 e _c5 — 1º semestre


valores_distintos_c4,valores_distintos_c5,registros_com_informacao_c4,registros_c5_diferente_zero
1087,1088,135291,135405


Perfil de _c4 e _c5 — 2º semestre


valores_distintos_c4,valores_distintos_c5,registros_com_informacao_c4,registros_c5_diferente_zero
1081,1082,134913,135027


In [0]:
# Duplicidades exatas existentes nos arquivos originais

colunas_originais_epb = [f"_c{i}" for i in range(16)]

def resumir_duplicidades_originais(df, semestre):
    grupos = (
        df
        .groupBy(*colunas_originais_epb)
        .count()
    )

    return grupos.agg(
        F.lit(semestre).alias("semestre"),
        F.sum("count").alias("registros_originais"),
        F.count("*").alias("linhas_distintas"),
        F.sum(
            F.when(F.col("count") > 1, F.col("count") - 1)
             .otherwise(0)
        ).alias("linhas_excedentes"),
        F.max("count").alias("maior_repeticao")
    )


df_duplicidades_originais = (
    resumir_duplicidades_originais(
        df_epb_1sem_bronze,
        "1º semestre"
    )
    .unionByName(
        resumir_duplicidades_originais(
            df_epb_2sem_bronze,
            "2º semestre"
        )
    )
)

display(df_duplicidades_originais)

semestre,registros_originais,linhas_distintas,linhas_excedentes,maior_repeticao
1º semestre,166225,166225,0,1
2º semestre,165661,165615,46,2


In [0]:
# Exemplos de registros mais repetidos na fonte

display(
    df_epb_1sem_bronze
    .groupBy(*colunas_originais_epb)
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
    .select(
        "_c0", "_c1", "_c2", "_c6", "_c7", "_c8",
        "_c9", "_c10", "_c11", "_c13", "_c14",
        "_c15", "count"
    )
    .limit(30)
)

_c0,_c1,_c2,_c6,_c7,_c8,_c9,_c10,_c11,_c13,_c14,_c15,count


In [0]:
# Identificação dos campos _c4 e _c5

display(
    df_epb_1sem_bronze
    .filter(
        F.col("_c4").isNotNull()
        & (F.upper(F.trim(F.col("_c4"))) != "NULL")
    )
    .select(
        "_c0",
        "_c1",
        "_c2",
        "_c3",
        "_c4",
        "_c5",
        "_c6",
        "_c7",
        "_c8"
    )
    .distinct()
    .limit(50)
)

_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8
202501,941,SERPROS,PLANO,185,1980001618,7,12000,Auxílios - Prestação Continuada
202501,941,SERPROS,PLANO,185,1980001618,21,31200,Participante - com custeio patronal e do participante
202501,941,SERPROS,PLANO,185,1980001618,23,32000,Assistidos - Aposentados
202501,941,SERPROS,PLANO,185,1980001618,24,33000,Assistidos - Beneficiários de Pensão
202501,941,SERPROS,PLANO,185,1980001618,25,34000,Designados
202501,941,SERPROS,PLANO,186,1998007774,7,12000,Auxílios - Prestação Continuada
202501,941,SERPROS,PLANO,186,1998007774,21,31200,Participante - com custeio patronal e do participante
202501,941,SERPROS,PLANO,186,1998007774,23,32000,Assistidos - Aposentados
202501,941,SERPROS,PLANO,186,1998007774,24,33000,Assistidos - Beneficiários de Pensão
202501,941,SERPROS,PLANO,186,1998007774,25,34000,Designados


In [0]:
# Relação entre entidade e os campos _c4 e _c5

display(
    df_epb_1sem_bronze
    .filter(
        F.col("_c4").isNotNull()
        & (F.upper(F.trim(F.col("_c4"))) != "NULL")
    )
    .groupBy("_c1", "_c2")
    .agg(
        F.countDistinct("_c4").alias("quantidade_valores_c4"),
        F.countDistinct("_c5").alias("quantidade_valores_c5"),
        F.first("_c4").alias("exemplo_c4"),
        F.first("_c5").alias("exemplo_c5")
    )
    .orderBy(F.desc("quantidade_valores_c4"))
    .limit(30)
)

_c1,_c2,quantidade_valores_c4,quantidade_valores_c5,exemplo_c4,exemplo_c5
2258,MULTIPREV,94,94,553,1995000183
1482,MULTIBRA,78,78,372,2001000138
3438,ICATUFMP,45,45,833,1999001011
3126,IFM,42,42,384,1993001492
3825,MULTIPENSIONS,40,40,823,1993001638
3188,BB PREVIDENCIA,40,40,683,1999002792
1239,FUNCESP,25,25,682,1998007383
655,PETROS,23,23,5645,2018000292
1911,PREVISC,18,18,1115,2007001292
611,ITAU UNIBANCO,18,18,112,1990000347


In [0]:
# Duplicatas exatas existentes no segundo semestre

display(
    df_epb_2sem_bronze
    .groupBy(*colunas_originais_epb)
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
    .select(
        "_c0", "_c1", "_c2", "_c3", "_c4", "_c5",
        "_c6", "_c7", "_c8", "_c9",
        "_c10", "_c11", "_c13", "_c14", "_c15",
        "count"
    )
    .limit(50)
)

_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9,_c10,_c11,_c13,_c14,_c15,count
202512,2169,BRASILETROS,EFPC,NULL,0,4,11000,Aposentadoria - Prestação Continuada (totalizador),TOTALIZADOR,1462,1,4,1459,2026-06-26,2
202512,3825,MULTIPENSIONS,PLANO,7186,2020003211,14,22000,Autopatrocínio,FOLHA,3,0,0,3,2026-06-26,2
202511,3825,MULTIPENSIONS,PLANO,7186,2020003211,6,11200,Aposentadoria por Invalidez,FOLHA,0,0,0,0,2026-06-26,2
202511,3825,MULTIPENSIONS,PLANO,7186,2020003211,8,13000,Auxílios - Prestação Única,FOLHA,0,0,0,0,2026-06-26,2
202511,3825,MULTIPENSIONS,PLANO,7186,2020003211,25,34000,Designados,FOLHA,751,1,1,751,2026-06-26,2
202512,3825,MULTIPENSIONS,PLANO,7186,2020003211,6,11200,Aposentadoria por Invalidez,FOLHA,0,0,0,0,2026-06-26,2
202511,3825,MULTIPENSIONS,PLANO,7186,2020003211,11,16000,Outros Benefícios de Prestação Única,FOLHA,29,1,0,30,2026-06-26,2
202511,3825,MULTIPENSIONS,PLANO,7186,2020003211,22,31300,Participante - com custeio exclusivamente do participante,FOLHA,96,0,0,96,2026-06-26,2
202511,3825,MULTIPENSIONS,PLANO,7186,2020003211,15,23000,Resgate,FOLHA,82,1,0,83,2026-06-26,2
202512,3825,MULTIPENSIONS,PLANO,7186,2020003211,8,13000,Auxílios - Prestação Única,FOLHA,0,0,0,0,2026-06-26,2


In [0]:
# Função corrigida para tratamento dos arquivos EPB

def preparar_epb_silver(df, semestre_origem):
    
    # Remove somente duplicatas integrais existentes no arquivo original
    df_sem_duplicatas = df.dropDuplicates(
        [f"_c{i}" for i in range(16)]
    )

    return (
        df_sem_duplicatas
        .select(
            F.trim(F.col("_c0")).alias("ano_mes"),
            F.trim(F.col("_c1")).cast("int").alias("codigo_entidade"),
            F.upper(F.trim(F.col("_c2"))).alias("sigla_entidade"),
            F.upper(F.trim(F.col("_c3"))).alias("nivel_informacao"),

            # Código interno do plano
            F.when(
                F.upper(F.trim(F.col("_c4"))) != "NULL",
                F.trim(F.col("_c4")).cast("int")
            ).otherwise(F.lit(None).cast("int"))
            .alias("codigo_plano"),

            # CNPB preservado como texto
            F.when(
                F.trim(F.col("_c5")) != "0",
                F.trim(F.col("_c5"))
            ).otherwise(F.lit(None).cast("string"))
            .alias("cnpb_plano"),

            F.trim(F.col("_c6")).cast("int")
                .alias("codigo_tipo_informacao"),

            F.trim(F.col("_c7")).cast("int")
                .alias("codigo_conta"),

            F.trim(F.col("_c8")).alias("descricao_conta"),

            F.upper(F.trim(F.col("_c9")))
                .alias("origem_informacao"),

            F.trim(F.col("_c10")).cast("long")
                .alias("quantidade_inicial_informada"),

            F.trim(F.col("_c11")).cast("long")
                .alias("entradas_informadas"),

            # _c12 foi descartado porque é idêntico a _c13
            F.trim(F.col("_c13")).cast("long")
                .alias("saidas_informadas"),

            F.trim(F.col("_c14")).cast("long")
                .alias("quantidade_final_informada"),

            F.to_date(
                F.trim(F.col("_c15")),
                "yyyy-MM-dd"
            ).alias("data_atualizacao_fonte"),

            F.lit(semestre_origem).alias("semestre_origem"),
            F.col("_arquivo_origem"),
            F.col("_data_ingestao")
        )
        .withColumn(
            "data_referencia",
            F.to_date(
                F.concat(F.col("ano_mes"), F.lit("01")),
                "yyyyMMdd"
            )
        )
        .filter(F.col("ano_mes") <= "202512")
        .withColumn(
            "flag_entrada_invalida",
            F.coalesce(
                F.col("entradas_informadas") < 0,
                F.lit(False)
            )
        )
        .withColumn(
            "flag_quantidade_final_invalida",
            F.coalesce(
                F.col("quantidade_final_informada") < 0,
                F.lit(False)
            )
        )
        .withColumn(
            "flag_registro_invalido",
            F.col("flag_entrada_invalida")
            | F.col("flag_quantidade_final_invalida")
        )
        .withColumn(
            "quantidade_inicial",
            F.when(
                F.col("quantidade_inicial_informada") >= 0,
                F.col("quantidade_inicial_informada")
            ).otherwise(F.lit(None).cast("long"))
        )
        .withColumn(
            "entradas",
            F.when(
                F.col("entradas_informadas") >= 0,
                F.col("entradas_informadas")
            ).otherwise(F.lit(None).cast("long"))
        )
        .withColumn(
            "saidas",
            F.when(
                F.col("saidas_informadas") >= 0,
                F.col("saidas_informadas")
            ).otherwise(F.lit(None).cast("long"))
        )
        .withColumn(
            "quantidade_final",
            F.when(
                F.col("quantidade_final_informada") >= 0,
                F.col("quantidade_final_informada")
            ).otherwise(F.lit(None).cast("long"))
        )
    )


# Reconstrução e consolidação corrigida

df_epb_1sem_silver = preparar_epb_silver(
    df_epb_1sem_bronze,
    "1º semestre"
)

df_epb_2sem_silver = preparar_epb_silver(
    df_epb_2sem_bronze,
    "2º semestre"
)

df_epb_silver = (
    df_epb_1sem_silver
    .unionByName(df_epb_2sem_silver)
    .select(
        "data_referencia",
        "ano_mes",
        "codigo_entidade",
        "sigla_entidade",
        "nivel_informacao",
        "codigo_plano",
        "cnpb_plano",
        "codigo_tipo_informacao",
        "codigo_conta",
        "descricao_conta",
        "origem_informacao",
        "quantidade_inicial",
        "entradas",
        "saidas",
        "quantidade_final",
        "quantidade_inicial_informada",
        "entradas_informadas",
        "saidas_informadas",
        "quantidade_final_informada",
        "flag_entrada_invalida",
        "flag_quantidade_final_invalida",
        "flag_registro_invalido",
        "data_atualizacao_fonte",
        "semestre_origem",
        "_arquivo_origem",
        "_data_ingestao"
    )
)

display(df_epb_silver.limit(20))

data_referencia,ano_mes,codigo_entidade,sigla_entidade,nivel_informacao,codigo_plano,cnpb_plano,codigo_tipo_informacao,codigo_conta,descricao_conta,origem_informacao,quantidade_inicial,entradas,saidas,quantidade_final,quantidade_inicial_informada,entradas_informadas,saidas_informadas,quantidade_final_informada,flag_entrada_invalida,flag_quantidade_final_invalida,flag_registro_invalido,data_atualizacao_fonte,semestre_origem,_arquivo_origem,_data_ingestao
2025-01-01,202501,941,SERPROS,EFPC,null,null,21,31200,Participante - com custeio patronal e do participante,FOLHA,7466,11,129,7348,7466,11,129,7348,false,false,false,2026-06-26,1º semestre,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
2025-06-01,202506,941,SERPROS,EFPC,null,null,7,12000,Auxílios - Prestação Continuada,FOLHA,11,12,1,22,11,12,1,22,false,false,false,2026-06-26,1º semestre,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
2025-03-01,202503,941,SERPROS,PLANO,185,1980001618,21,31200,Participante - com custeio patronal e do participante,FOLHA,1062,0,11,1051,1062,0,11,1051,false,false,false,2026-06-26,1º semestre,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
2025-03-01,202503,941,SERPROS,PLANO,186,1998007774,23,32000,Assistidos - Aposentados,FOLHA,995,6,1,1000,995,6,1,1000,false,false,false,2026-06-26,1º semestre,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
2025-04-01,202504,941,SERPROS,PLANO,185,1980001618,23,32000,Assistidos - Aposentados,FOLHA,3875,6,4,3877,3875,6,4,3877,false,false,false,2026-06-26,1º semestre,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
2025-05-01,202505,941,SERPROS,PLANO,186,1998007774,25,34000,Designados,FOLHA,14414,26,2,14438,14414,26,2,14438,false,false,false,2026-06-26,1º semestre,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
2025-04-01,202504,941,SERPROS,EFPC,null,null,19,31000,Participantes Ativos,TOTALIZADOR,7573,30,24,7579,7573,30,24,7579,false,false,false,2026-06-26,1º semestre,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
2025-06-01,202506,941,SERPROS,EFPC,null,null,19,31000,Participantes Ativos,TOTALIZADOR,7600,31,7,7624,7600,31,7,7624,false,false,false,2026-06-26,1º semestre,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
2025-03-01,202503,941,SERPROS,PLANO,186,1998007774,20,31100,Participante - com custeio exclusivamente patronal,FOLHA,0,0,0,0,0,0,0,0,false,false,false,2026-06-26,1º semestre,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z
2025-04-01,202504,941,SERPROS,PLANO,185,1980001618,13,21000,Benefício Proporcional Diferido,FOLHA,48,0,5,43,48,0,5,43,false,false,false,2026-06-26,1º semestre,EPB_1SEMESTRE_2025.csv,2026-09-07T02:23:21.880Z


In [0]:
print(f"Total após tratamento: {df_epb_silver.count()}")

display(
    df_epb_silver
    .groupBy("nivel_informacao")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("codigo_plano").alias("planos_distintos"),
        F.countDistinct("cnpb_plano").alias("cnpbs_distintos")
    )
)

Total após tratamento: 331840


nivel_informacao,registros,planos_distintos,cnpbs_distintos
PLANO,270387,1091,1092
EFPC,61453,0,0


In [0]:
# Validação dos identificadores dos registros de planos

df_validacao_identificadores_epb = (
    df_epb_silver
    .filter(F.col("nivel_informacao") == "PLANO")
    .agg(
        F.count("*").alias("registros_plano"),
        F.sum(
            F.when(F.col("codigo_plano").isNull(), 1).otherwise(0)
        ).alias("codigos_plano_nulos"),
        F.sum(
            F.when(F.col("cnpb_plano").isNull(), 1).otherwise(0)
        ).alias("cnpbs_nulos"),
        F.countDistinct("codigo_plano").alias("planos_distintos"),
        F.countDistinct("cnpb_plano").alias("cnpbs_distintos")
    )
)

display(df_validacao_identificadores_epb)

registros_plano,codigos_plano_nulos,cnpbs_nulos,planos_distintos,cnpbs_distintos
270387,228,0,1091,1092


In [0]:
# Códigos de plano associados a mais de um CNPB

df_codigo_multiplos_cnpbs = (
    df_epb_silver
    .filter(
        (F.col("nivel_informacao") == "PLANO")
        & F.col("codigo_plano").isNotNull()
        & F.col("cnpb_plano").isNotNull()
    )
    .groupBy(
        "codigo_entidade",
        "sigla_entidade",
        "codigo_plano"
    )
    .agg(
        F.countDistinct("cnpb_plano").alias("quantidade_cnpbs"),
        F.collect_set("cnpb_plano").alias("cnpbs_encontrados"),
        F.min("ano_mes").alias("primeiro_periodo"),
        F.max("ano_mes").alias("ultimo_periodo")
    )
    .filter(F.col("quantidade_cnpbs") > 1)
)

display(df_codigo_multiplos_cnpbs)

codigo_entidade,sigla_entidade,codigo_plano,quantidade_cnpbs,cnpbs_encontrados,primeiro_periodo,ultimo_periodo


In [0]:
display(df_validacao_identificadores_epb)

registros_plano,codigos_plano_nulos,cnpbs_nulos,planos_distintos,cnpbs_distintos
270387,228,0,1091,1092


In [0]:
df_codigo_multiplos_cnpbs

DataFrame[codigo_entidade: int, sigla_entidade: string, codigo_plano: int, quantidade_cnpbs: bigint, cnpbs_encontrados: array<string>, primeiro_periodo: string, ultimo_periodo: string]

Foram identificados 228 registros de planos sem código interno, mas todos possuem CNPB preenchido. Esses registros foram preservados e sinalizados por meio de uma flag de qualidade. Não foram encontrados códigos de plano associados a múltiplos CNPBs dentro da mesma entidade.

In [0]:
# Sinalização dos registros de plano sem código interno

df_epb_silver = (
    df_epb_silver
    .withColumn(
        "flag_codigo_plano_ausente",
        (F.col("nivel_informacao") == "PLANO")
        & F.col("codigo_plano").isNull()
    )
)

In [0]:
# Gravação da tabela EPB consolidada na camada Silver

(
    df_epb_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.previc_epb_2025")
)

print("Tabela workspace.silver.previc_epb_2025 gravada com sucesso.")

Tabela workspace.silver.previc_epb_2025 gravada com sucesso.


In [0]:
# Validação final da tabela EPB após a gravação

df_epb_gravado = spark.table(
    "workspace.silver.previc_epb_2025"
)

df_validacao_final_epb = df_epb_gravado.agg(
    F.count("*").alias("quantidade_registros"),
    F.countDistinct("codigo_entidade").alias("quantidade_entidades"),
    F.countDistinct("cnpb_plano").alias("quantidade_cnpbs"),
    F.sum(
        F.when(F.col("flag_codigo_plano_ausente"), 1).otherwise(0)
    ).alias("codigos_plano_ausentes"),
    F.sum(
        F.when(F.col("flag_entrada_invalida"), 1).otherwise(0)
    ).alias("entradas_invalidas"),
    F.sum(
        F.when(
            F.col("flag_quantidade_final_invalida"), 1
        ).otherwise(0)
    ).alias("quantidades_finais_invalidas"),
    F.sum(
        F.when(F.col("flag_registro_invalido"), 1).otherwise(0)
    ).alias("registros_invalidos_sinalizados"),
    F.sum(
        F.when(F.col("data_referencia").isNull(), 1).otherwise(0)
    ).alias("datas_invalidas")
)

display(df_validacao_final_epb)

quantidade_registros,quantidade_entidades,quantidade_cnpbs,codigos_plano_ausentes,entradas_invalidas,quantidades_finais_invalidas,registros_invalidos_sinalizados,datas_invalidas
331840,246,1092,228,2,631,633,0


## 3. SUSEP — Previdência complementar aberta

As bases da SUSEP contêm informações sobre previdência tradicional, PGBL e VGBL, incluindo contribuições, benefícios, resgates, provisões, participantes, fundos e distribuição por unidade da Federação.

Diferentemente dos arquivos da PREVIC, as bases da SUSEP possuem cabeçalhos, mas apresentam estruturas, codificações e granularidades distintas. O tratamento será realizado por grupos de tabelas relacionadas, preservando os metadados técnicos e limitando os dados ao período de dezembro de 2025.

In [0]:
# Inventário das tabelas SUSEP existentes na camada Bronze

tabelas_susep_bronze = [
    "susep_lista_empresas",
    "susep_contrib_benef",
    "susep_vgbl_fundos",
    "susep_vgbl_resgates",
    "susep_pgbl_fundos",
    "susep_pgbl_resgates",
    "susep_pgbl_uf",
    "susep_prev_trad_resgates",
    "susep_prev_uf",
    "susep_prov_segprev",
    "susep_quantprev_benef",
    "susep_quantprev_part"
]

estrutura_susep = []

for nome_tabela in tabelas_susep_bronze:
    df_temp = spark.table(f"workspace.bronze.{nome_tabela}")
    
    colunas_origem = [
        coluna
        for coluna in df_temp.columns
        if coluna not in ["_arquivo_origem", "_data_ingestao"]
    ]
    
    estrutura_susep.append(
        (
            nome_tabela,
            len(colunas_origem),
            ", ".join(colunas_origem)
        )
    )

df_estrutura_susep = spark.createDataFrame(
    estrutura_susep,
    [
        "tabela",
        "quantidade_colunas_origem",
        "colunas_origem"
    ]
)

display(df_estrutura_susep)

tabela,quantidade_colunas_origem,colunas_origem
susep_lista_empresas,3,"CodigoFIP, NomeEntidade, CNPJ"
susep_contrib_benef,5,"coenti, damesano, tipoProd, contrib, benef"
susep_vgbl_fundos,3,"coenti, damesano, fundos"
susep_vgbl_resgates,5,"damesano, coenti, resg_total, resg_parcial, Resg_Pag_programado"
susep_pgbl_fundos,3,"coenti, damesano, fundos"
susep_pgbl_resgates,5,"damesano, coenti, resg_total, resg_parcial, Resg_Pag_programado"
susep_pgbl_uf,9,"COENTI, DAMESANO, UF, CONTRIB, BENEFPAGO, RESGPAGO, NUMPARTIC, NUMBENEF, NUMRESG"
susep_prev_trad_resgates,4,"damesano, coenti, resg_total, resg_parcial"
susep_prev_uf,9,"COENTI, DAMESANO, UF, CONTRIB, BENEFPAGO, RESGPAGO, NUMPARTIC, NUMBENEF, NUMRESG"
susep_prov_segprev,3,"coenti, damesano, valor"


### 3.1 Cadastro de entidades e contribuições/benefícios

O código FIP foi preservado como texto para manter zeros à esquerda. O CNPJ também foi mantido como texto, pois representa um identificador e não uma medida numérica.

Na base financeira, o período no formato `AAAAMM` foi convertido em data e os valores de contribuições e benefícios foram convertidos para decimal. Os textos originais foram preservados para permitir a identificação de eventuais falhas de conversão.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

tipo_monetario = DecimalType(20, 2)

# Cadastro de entidades SUSEP
df_susep_empresas_bronze = spark.table(
    "workspace.bronze.susep_lista_empresas"
)

df_susep_empresas_silver = (
    df_susep_empresas_bronze
    .select(
        F.lpad(
            F.trim(F.col("CodigoFIP")), 5, "0"
        ).alias("codigo_fip"),
        F.trim(F.col("NomeEntidade")).alias("nome_entidade"),
        F.lpad(
            F.trim(F.col("CNPJ")), 14, "0"
        ).alias("cnpj"),
        F.col("_arquivo_origem"),
        F.col("_data_ingestao")
    )
)

# Contribuições e benefícios
df_susep_contrib_benef_bronze = spark.table(
    "workspace.bronze.susep_contrib_benef"
)

df_susep_contrib_benef_silver = (
    df_susep_contrib_benef_bronze
    .select(
        F.lpad(
            F.trim(F.col("coenti")), 5, "0"
        ).alias("codigo_fip"),
        F.trim(F.col("damesano")).alias("ano_mes"),
        F.upper(
            F.trim(F.col("tipoProd"))
        ).alias("tipo_produto"),
        F.trim(F.col("contrib")).alias("contribuicao_informada"),
        F.trim(F.col("benef")).alias("beneficio_informado"),
        F.col("_arquivo_origem"),
        F.col("_data_ingestao")
    )
    .withColumn(
        "data_referencia",
        F.to_date(
            F.concat(F.col("ano_mes"), F.lit("01")),
            "yyyyMMdd"
        )
    )
    .withColumn(
        "contribuicao",
        F.regexp_replace(
            F.col("contribuicao_informada"), ",", "."
        ).cast(tipo_monetario)
    )
    .withColumn(
        "beneficio",
        F.regexp_replace(
            F.col("beneficio_informado"), ",", "."
        ).cast(tipo_monetario)
    )
    .filter(F.col("ano_mes") <= "202512")
    .withColumn(
        "flag_contribuicao_nao_convertida",
        F.col("contribuicao_informada").isNotNull()
        & (F.trim(F.col("contribuicao_informada")) != "")
        & F.col("contribuicao").isNull()
    )
    .withColumn(
        "flag_beneficio_nao_convertido",
        F.col("beneficio_informado").isNotNull()
        & (F.trim(F.col("beneficio_informado")) != "")
        & F.col("beneficio").isNull()
    )
    .select(
        "data_referencia",
        "ano_mes",
        "codigo_fip",
        "tipo_produto",
        "contribuicao",
        "beneficio",
        "contribuicao_informada",
        "beneficio_informado",
        "flag_contribuicao_nao_convertida",
        "flag_beneficio_nao_convertido",
        "_arquivo_origem",
        "_data_ingestao"
    )
)

display(df_susep_empresas_silver.limit(20))

codigo_fip,nome_entidade,cnpj,_arquivo_origem,_data_ingestao
01007,SABEMI SEGURADORA S.A.,87163234000138,LISTAEMPRESAS.csv,2026-09-07T03:37:15.088Z
01121,YOUSE SEGURADORA S.A.,24856160000103,LISTAEMPRESAS.csv,2026-09-07T03:37:15.088Z
01414,Berkley International do Brasil Seguros S/A,07021544000189,LISTAEMPRESAS.csv,2026-09-07T03:37:15.088Z
01431,XL Seguros Brasil S.A.,14448493000131,LISTAEMPRESAS.csv,2026-09-07T03:37:15.088Z
01490,Essor Seguros S.A,14525684000150,LISTAEMPRESAS.csv,2026-09-07T03:37:15.088Z
01554,EQUATORIAL SEGURADORA S/A - MICROSSEGUROS,21242451000105,LISTAEMPRESAS.csv,2026-09-07T03:37:15.088Z
01571,HDI GLOBAL SEGUROS S.A.,18096627000153,LISTAEMPRESAS.csv,2026-09-07T03:37:15.088Z
01589,SANTANDER AUTO S.A.,30617319000121,LISTAEMPRESAS.csv,2026-09-07T03:37:15.088Z
01619,DAYPREV VIDA E PREVIDÊNCIA S.A.,08872199000150,LISTAEMPRESAS.csv,2026-09-07T03:37:15.088Z
01627,SAFRA SEGUROS GERAIS S.A.,06109373000181,LISTAEMPRESAS.csv,2026-09-07T03:37:15.088Z


In [0]:
display(df_susep_contrib_benef_silver.limit(20))

data_referencia,ano_mes,codigo_fip,tipo_produto,contribuicao,beneficio,contribuicao_informada,beneficio_informado,flag_contribuicao_nao_convertida,flag_beneficio_nao_convertido,_arquivo_origem,_data_ingestao
2023-03-01,202303,06084,PREVTRAD,567708.61,576537.48,"567708,61","576537,48",false,false,Ses_Contrib_Benef.csv,2026-09-07T03:37:23.797Z
2007-10-01,200710,05096,PREVTRAD,10278813.95,3995111.04,"10278813,95","3995111,04",false,false,Ses_Contrib_Benef.csv,2026-09-07T03:37:23.797Z
2005-02-01,200502,05096,VGBL,15695528.28,127276.60,"15695528,28","127276,6",false,false,Ses_Contrib_Benef.csv,2026-09-07T03:37:23.797Z
2019-05-01,201905,09938,VGBL,96189039.43,23616.78,"96189039,43","23616,78",false,false,Ses_Contrib_Benef.csv,2026-09-07T03:37:23.797Z
2004-06-01,200406,08141,PGBL,24852826.44,21610.55,"24852826,44","21610,55",false,false,Ses_Contrib_Benef.csv,2026-09-07T03:37:23.797Z
2005-11-01,200511,10979,PREVTRAD,88562.33,26040.58,"88562,33","26040,58",false,false,Ses_Contrib_Benef.csv,2026-09-07T03:37:23.797Z
2021-06-01,202106,10448,PGBL,7332.78,0.00,"7332,78",0,false,false,Ses_Contrib_Benef.csv,2026-09-07T03:37:23.797Z
2016-08-01,201608,06220,VGBL,18300715.25,721473.54,"18300715,25","721473,54",false,false,Ses_Contrib_Benef.csv,2026-09-07T03:37:23.797Z
2018-05-01,201805,02895,PGBL,1100951.38,27444.76,"1100951,38","27444,76",false,false,Ses_Contrib_Benef.csv,2026-09-07T03:37:23.797Z
2002-11-01,200211,05215,PREVTRAD,25031142.96,411896.16,"25031142,96","411896,16",false,false,Ses_Contrib_Benef.csv,2026-09-07T03:37:23.797Z


In [0]:
# Validação das primeiras bases SUSEP

df_validacao_contrib_benef = (
    df_susep_contrib_benef_silver
    .agg(
        F.count("*").alias("quantidade_registros"),
        F.min("ano_mes").alias("primeiro_periodo"),
        F.max("ano_mes").alias("ultimo_periodo"),
        F.countDistinct("codigo_fip").alias("entidades_distintas"),
        F.countDistinct("tipo_produto").alias("tipos_produto"),
        F.sum(
            F.when(
                F.col("flag_contribuicao_nao_convertida"), 1
            ).otherwise(0)
        ).alias("contribuicoes_nao_convertidas"),
        F.sum(
            F.when(
                F.col("flag_beneficio_nao_convertido"), 1
            ).otherwise(0)
        ).alias("beneficios_nao_convertidos"),
        F.sum(
            F.when(F.col("contribuicao") < 0, 1).otherwise(0)
        ).alias("contribuicoes_negativas"),
        F.sum(
            F.when(F.col("beneficio") < 0, 1).otherwise(0)
        ).alias("beneficios_negativos"),
        F.sum(
            F.when(F.col("data_referencia").isNull(), 1).otherwise(0)
        ).alias("datas_invalidas")
    )
)

display(df_validacao_contrib_benef)

quantidade_registros,primeiro_periodo,ultimo_periodo,entidades_distintas,tipos_produto,contribuicoes_nao_convertidas,beneficios_nao_convertidos,contribuicoes_negativas,beneficios_negativos,datas_invalidas
27605,199906,202512,124,3,0,0,22,3,0


In [0]:
display(
    df_susep_contrib_benef_silver
    .groupBy("tipo_produto")
    .count()
    .orderBy("tipo_produto")
)

tipo_produto,count
PGBL,6889
PREVTRAD,14777
VGBL,5939


In [0]:
# Investigação dos valores financeiros negativos

display(
    df_susep_contrib_benef_silver
    .filter(
        (F.col("contribuicao") < 0)
        | (F.col("beneficio") < 0)
    )
    .select(
        "data_referencia",
        "codigo_fip",
        "tipo_produto",
        "contribuicao",
        "beneficio",
        "contribuicao_informada",
        "beneficio_informado"
    )
    .orderBy(
        "data_referencia",
        "codigo_fip",
        "tipo_produto"
    )
)

data_referencia,codigo_fip,tipo_produto,contribuicao,beneficio,contribuicao_informada,beneficio_informado
1999-09-01,05487,PREVTRAD,715394.00,-105966.00,715394,-105966
2000-03-01,06190,PREVTRAD,8364926.00,-14706.00,8364926,-14706
2000-03-01,06904,PREVTRAD,-29973.00,0.00,-29973,0
2000-05-01,06939,PREVTRAD,684189.00,-165.00,684189,-165
2002-06-01,05070,PGBL,-3103445.09,0.00,"-3103445,09",0
2004-08-01,06220,PREVTRAD,-29890204.73,2188538.77,"-29890204,73","2188538,77"
2004-09-01,06220,PREVTRAD,-7406432.53,2283250.89,"-7406432,53","2283250,89"
2004-10-01,06220,PREVTRAD,-2226985.40,2174685.07,"-2226985,4","2174685,07"
2009-04-01,05096,PGBL,-58177606.26,2759936.45,"-58177606,2600001","2759936,45"
2013-06-01,06173,PREVTRAD,-1829838.45,276151.13,"-1829838,45","276151,13"


In [0]:
# Sinalização de valores negativos, preservando possíveis ajustes

df_susep_contrib_benef_silver = (
    df_susep_contrib_benef_silver
    .withColumn(
        "flag_contribuicao_negativa",
        F.col("contribuicao") < 0
    )
    .withColumn(
        "flag_beneficio_negativo",
        F.col("beneficio") < 0
    )
    .withColumn(
        "flag_valor_negativo",
        F.col("flag_contribuicao_negativa")
        | F.col("flag_beneficio_negativo")
    )
)

In [0]:
# Entidades financeiras sem correspondência no cadastro

df_entidades_sem_cadastro = (
    df_susep_contrib_benef_silver
    .select("codigo_fip")
    .distinct()
    .join(
        df_susep_empresas_silver.select("codigo_fip").distinct(),
        on="codigo_fip",
        how="left_anti"
    )
)

print(
    "Entidades sem correspondência no cadastro:",
    df_entidades_sem_cadastro.count()
)

display(df_entidades_sem_cadastro)

Entidades sem correspondência no cadastro: 57


codigo_fip
06939
10669
06149
05215
10634
10359
05801
10405
10014
04561


In [0]:
# Conferência de duplicatas exatas

colunas_conteudo_contrib_benef = [
    "ano_mes",
    "codigo_fip",
    "tipo_produto",
    "contribuicao_informada",
    "beneficio_informado"
]

total_contrib_benef = df_susep_contrib_benef_silver.count()

total_distinto_contrib_benef = (
    df_susep_contrib_benef_silver
    .select(*colunas_conteudo_contrib_benef)
    .distinct()
    .count()
)

print(f"Total de registros: {total_contrib_benef}")
print(f"Linhas distintas: {total_distinto_contrib_benef}")
print(
    "Duplicatas exatas:",
    total_contrib_benef - total_distinto_contrib_benef
)

Total de registros: 27605
Linhas distintas: 27605
Duplicatas exatas: 0


A tabela financeira contém séries históricas desde 1999. Foram encontrados códigos de entidades que não constam na lista cadastral atual da SUSEP. Esses registros foram preservados, pois podem representar entidades encerradas, incorporadas ou com alteração cadastral.

Também foram identificados valores negativos de contribuições e benefícios. Como valores financeiros negativos podem representar ajustes ou estornos, eles foram mantidos e sinalizados, sem substituição por zero ou nulo.

In [0]:
# Sinalização de entidades históricas ausentes no cadastro atual

df_codigos_susep_cadastrados = (
    df_susep_empresas_silver
    .select("codigo_fip")
    .distinct()
    .withColumn("consta_cadastro_atual", F.lit(True))
)

df_susep_contrib_benef_silver = (
    df_susep_contrib_benef_silver
    .join(
        df_codigos_susep_cadastrados,
        on="codigo_fip",
        how="left"
    )
    .withColumn(
        "flag_entidade_sem_cadastro_atual",
        F.col("consta_cadastro_atual").isNull()
    )
    .drop("consta_cadastro_atual")
)

In [0]:
# Qualidade do cadastro de empresas

df_validacao_empresas = df_susep_empresas_silver.agg(
    F.count("*").alias("quantidade_registros"),
    F.countDistinct("codigo_fip").alias("codigos_fip_distintos"),
    F.countDistinct("cnpj").alias("cnpjs_distintos"),
    F.sum(
        F.when(
            (F.length("codigo_fip") != 5)
            | (~F.col("codigo_fip").rlike("^[0-9]{5}$")),
            1
        ).otherwise(0)
    ).alias("codigos_fip_invalidos"),
    F.sum(
        F.when(
            (F.length("cnpj") != 14)
            | (~F.col("cnpj").rlike("^[0-9]{14}$")),
            1
        ).otherwise(0)
    ).alias("cnpjs_invalidos"),
    F.sum(
        F.when(
            F.col("nome_entidade").isNull()
            | (F.trim(F.col("nome_entidade")) == ""),
            1
        ).otherwise(0)
    ).alias("nomes_ausentes")
)

display(df_validacao_empresas)

quantidade_registros,codigos_fip_distintos,cnpjs_distintos,codigos_fip_invalidos,cnpjs_invalidos,nomes_ausentes
233,233,232,0,0,0


In [0]:
# CNPJ associado a mais de um código FIP

df_cnpj_multiplos_codigos = (
    df_susep_empresas_silver
    .groupBy("cnpj")
    .agg(
        F.countDistinct("codigo_fip").alias("quantidade_codigos_fip"),
        F.collect_set("codigo_fip").alias("codigos_fip"),
        F.collect_set("nome_entidade").alias("nomes_entidade")
    )
    .filter(F.col("quantidade_codigos_fip") > 1)
)

display(df_cnpj_multiplos_codigos)

cnpj,quantidade_codigos_fip,codigos_fip,nomes_entidade


In [0]:
# Identificação de empresas com CNPJ ausente

df_empresas_cnpj_ausente = (
    df_susep_empresas_silver
    .filter(
        F.col("cnpj").isNull()
        | (F.trim(F.col("cnpj")) == "")
    )
)

display(df_empresas_cnpj_ausente)

codigo_fip,nome_entidade,cnpj,_arquivo_origem,_data_ingestao
48186,MÜNCHENER RÜCKVERSICHERUNGS-GESELLSCHAFT AKTIENGESELLSCHAFT IN MÜNCHEN,null,LISTAEMPRESAS.csv,2026-09-07T03:37:15.088Z


In [0]:
# Sinalização de CNPJ ausente

df_susep_empresas_silver = (
    df_susep_empresas_silver
    .withColumn(
        "flag_cnpj_ausente",
        F.col("cnpj").isNull()
        | (F.trim(F.col("cnpj")) == "")
    )
)

In [0]:
# Sinalização de CNPJ ausente
display(
    df_susep_empresas_silver.agg(
        F.count("*").alias("quantidade_registros"),
        F.countDistinct("codigo_fip").alias("codigos_fip_distintos"),
        F.countDistinct("cnpj").alias("cnpjs_distintos"),
        F.sum(
            F.when(F.col("flag_cnpj_ausente"), 1).otherwise(0)
        ).alias("cnpjs_ausentes"),
        F.sum(
            F.when(
                F.col("cnpj").isNotNull()
                & (
                    (F.length("cnpj") != 14)
                    | (~F.col("cnpj").rlike("^[0-9]{14}$"))
                ),
                1
            ).otherwise(0)
        ).alias("cnpjs_com_formato_invalido")
    )
)

quantidade_registros,codigos_fip_distintos,cnpjs_distintos,cnpjs_ausentes,cnpjs_com_formato_invalido
233,233,232,1,0


In [0]:
# Gravação das primeiras tabelas SUSEP na Silver

(
    df_susep_empresas_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.susep_lista_empresas")
)

(
    df_susep_contrib_benef_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.susep_contrib_benef")
)

print("Tabelas SUSEP gravadas com sucesso.")

Tabelas SUSEP gravadas com sucesso.


In [0]:
# Validação da gravação das primeiras tabelas SUSEP na Silver
df_validacao_gravacao_susep = spark.createDataFrame(
    [
        (
            "susep_lista_empresas",
            spark.table(
                "workspace.silver.susep_lista_empresas"
            ).count()
        ),
        (
            "susep_contrib_benef",
            spark.table(
                "workspace.silver.susep_contrib_benef"
            ).count()
        )
    ],
    ["tabela", "quantidade_registros"]
)

display(df_validacao_gravacao_susep)

tabela,quantidade_registros
susep_lista_empresas,233
susep_contrib_benef,27605


No cadastro atual da SUSEP, uma entidade apresentou CNPJ ausente. O registro foi preservado porque possui código FIP e nome da entidade, sendo criada uma flag específica para documentar a ausência.

### 3.2 Fundos e resgates de PGBL e VGBL

As bases nacionais de PGBL e VGBL possuem estruturas equivalentes e foram consolidadas por assunto. Uma coluna de produto foi adicionada para preservar a identificação da origem.

Os campos monetários foram convertidos para decimal, mantendo-se também o texto original. Foram preservados somente os períodos até dezembro de 2025.

In [0]:
# Funções de tratamento das bases nacionais de fundos e resgates

def preparar_fundos_susep(nome_tabela, produto):
    df = spark.table(f"workspace.bronze.{nome_tabela}")

    return (
        df
        .select(
            F.lpad(F.trim(F.col("coenti")), 5, "0").alias("codigo_fip"),
            F.trim(F.col("damesano")).alias("ano_mes"),
            F.trim(F.col("fundos")).alias("valor_fundos_informado"),
            F.lit(produto).alias("tipo_produto"),
            F.col("_arquivo_origem"),
            F.col("_data_ingestao")
        )
        .withColumn(
            "data_referencia",
            F.to_date(
                F.concat(F.col("ano_mes"), F.lit("01")),
                "yyyyMMdd"
            )
        )
        .withColumn(
            "valor_fundos",
            F.regexp_replace(
                F.col("valor_fundos_informado"), ",", "."
            ).cast(tipo_monetario)
        )
        .filter(F.col("ano_mes") <= "202512")
        .withColumn(
            "flag_valor_nao_convertido",
            F.col("valor_fundos_informado").isNotNull()
            & (F.trim(F.col("valor_fundos_informado")) != "")
            & F.col("valor_fundos").isNull()
        )
        .withColumn(
            "flag_valor_negativo",
            F.col("valor_fundos") < 0
        )
    )


def preparar_resgates_susep(nome_tabela, produto):
    df = spark.table(f"workspace.bronze.{nome_tabela}")

    return (
        df
        .select(
            F.lpad(F.trim(F.col("coenti")), 5, "0").alias("codigo_fip"),
            F.trim(F.col("damesano")).alias("ano_mes"),
            F.trim(F.col("resg_total"))
                .alias("resgate_total_informado"),
            F.trim(F.col("resg_parcial"))
                .alias("resgate_parcial_informado"),
            F.trim(F.col("Resg_Pag_programado"))
                .alias("resgate_programado_informado"),
            F.lit(produto).alias("tipo_produto"),
            F.col("_arquivo_origem"),
            F.col("_data_ingestao")
        )
        .withColumn(
            "data_referencia",
            F.to_date(
                F.concat(F.col("ano_mes"), F.lit("01")),
                "yyyyMMdd"
            )
        )
        .withColumn(
            "resgate_total",
            F.regexp_replace(
                F.col("resgate_total_informado"), ",", "."
            ).cast(tipo_monetario)
        )
        .withColumn(
            "resgate_parcial",
            F.regexp_replace(
                F.col("resgate_parcial_informado"), ",", "."
            ).cast(tipo_monetario)
        )
        .withColumn(
            "resgate_programado",
            F.regexp_replace(
                F.col("resgate_programado_informado"), ",", "."
            ).cast(tipo_monetario)
        )
        .filter(F.col("ano_mes") <= "202512")
        .withColumn(
            "flag_valor_nao_convertido",
            (
                F.col("resgate_total_informado").isNotNull()
                & F.col("resgate_total").isNull()
            )
            | (
                F.col("resgate_parcial_informado").isNotNull()
                & F.col("resgate_parcial").isNull()
            )
            | (
                F.col("resgate_programado_informado").isNotNull()
                & F.col("resgate_programado").isNull()
            )
        )
        .withColumn(
            "flag_valor_negativo",
            (F.col("resgate_total") < 0)
            | (F.col("resgate_parcial") < 0)
            | (F.col("resgate_programado") < 0)
        )
    )

In [0]:
# Consolidação por assunto

df_susep_fundos_silver = (
    preparar_fundos_susep("susep_pgbl_fundos", "PGBL")
    .unionByName(
        preparar_fundos_susep("susep_vgbl_fundos", "VGBL")
    )
)

df_susep_resgates_silver = (
    preparar_resgates_susep("susep_pgbl_resgates", "PGBL")
    .unionByName(
        preparar_resgates_susep("susep_vgbl_resgates", "VGBL")
    )
)

In [0]:
df_susep_fundos_silver = (
    df_susep_fundos_silver
    .join(
        df_codigos_susep_cadastrados,
        on="codigo_fip",
        how="left"
    )
    .withColumn(
        "flag_entidade_sem_cadastro_atual",
        F.col("consta_cadastro_atual").isNull()
    )
    .drop("consta_cadastro_atual")
)

df_susep_resgates_silver = (
    df_susep_resgates_silver
    .join(
        df_codigos_susep_cadastrados,
        on="codigo_fip",
        how="left"
    )
    .withColumn(
        "flag_entidade_sem_cadastro_atual",
        F.col("consta_cadastro_atual").isNull()
    )
    .drop("consta_cadastro_atual")
)

In [0]:
# Resumo de qualidade das bases consolidadas

df_validacao_fundos = df_susep_fundos_silver.agg(
    F.count("*").alias("registros"),
    F.min("ano_mes").alias("primeiro_periodo"),
    F.max("ano_mes").alias("ultimo_periodo"),
    F.sum(
        F.when(F.col("flag_valor_nao_convertido"), 1).otherwise(0)
    ).alias("valores_nao_convertidos"),
    F.sum(
        F.when(F.col("flag_valor_negativo"), 1).otherwise(0)
    ).alias("valores_negativos"),
    F.sum(
        F.when(
            F.col("flag_entidade_sem_cadastro_atual"), 1
        ).otherwise(0)
    ).alias("registros_sem_cadastro")
)

df_validacao_resgates = df_susep_resgates_silver.agg(
    F.count("*").alias("registros"),
    F.min("ano_mes").alias("primeiro_periodo"),
    F.max("ano_mes").alias("ultimo_periodo"),
    F.sum(
        F.when(F.col("flag_valor_nao_convertido"), 1).otherwise(0)
    ).alias("valores_nao_convertidos"),
    F.sum(
        F.when(F.col("flag_valor_negativo"), 1).otherwise(0)
    ).alias("valores_negativos"),
    F.sum(
        F.when(
            F.col("flag_entidade_sem_cadastro_atual"), 1
        ).otherwise(0)
    ).alias("registros_sem_cadastro")
)

print("Validação dos fundos:")
display(df_validacao_fundos)

print("Validação dos resgates:")
display(df_validacao_resgates)

Validação dos fundos:


registros,primeiro_periodo,ultimo_periodo,valores_nao_convertidos,valores_negativos,registros_sem_cadastro
12992,200101,202512,0,0,1923


Validação dos resgates:


registros,primeiro_periodo,ultimo_periodo,valores_nao_convertidos,valores_negativos,registros_sem_cadastro
36041,200401,202512,0,0,1963


In [0]:
# Conferência final das bases de fundos e resgates

duplicatas_fundos = (
    df_susep_fundos_silver
    .groupBy(
        "ano_mes",
        "codigo_fip",
        "tipo_produto",
        "valor_fundos_informado"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicatas_resgates = (
    df_susep_resgates_silver
    .groupBy(
        "ano_mes",
        "codigo_fip",
        "tipo_produto",
        "resgate_total_informado",
        "resgate_parcial_informado",
        "resgate_programado_informado"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicatas exatas em fundos: {duplicatas_fundos}")
print(f"Duplicatas exatas em resgates: {duplicatas_resgates}")

print("Registros de fundos por produto:")
display(
    df_susep_fundos_silver
    .groupBy("tipo_produto")
    .count()
    .orderBy("tipo_produto")
)

print("Registros de resgates por produto:")
display(
    df_susep_resgates_silver
    .groupBy("tipo_produto")
    .count()
    .orderBy("tipo_produto")
)

Duplicatas exatas em fundos: 0
Duplicatas exatas em resgates: 5918
Registros de fundos por produto:


tipo_produto,count
PGBL,6969
VGBL,6023


Registros de resgates por produto:


tipo_produto,count
PGBL,17250
VGBL,18791


In [0]:
# Diagnóstico detalhado das duplicatas de resgates

colunas_resgates_conteudo = [
    "ano_mes",
    "codigo_fip",
    "tipo_produto",
    "resgate_total_informado",
    "resgate_parcial_informado",
    "resgate_programado_informado"
]

df_grupos_duplicados_resgates = (
    df_susep_resgates_silver
    .groupBy(*colunas_resgates_conteudo)
    .count()
    .filter(F.col("count") > 1)
)

df_resumo_duplicatas_resgates = (
    df_grupos_duplicados_resgates
    .agg(
        F.count("*").alias("grupos_duplicados"),
        F.sum(F.col("count") - 1).alias("linhas_excedentes"),
        F.max("count").alias("maior_repeticao")
    )
)

display(df_resumo_duplicatas_resgates)

grupos_duplicados,linhas_excedentes,maior_repeticao
5918,14259,10


In [0]:
#verificando os casos que mais se repetem:
display(
    df_grupos_duplicados_resgates
    .orderBy(F.desc("count"))
    .limit(30)
)

ano_mes,codigo_fip,tipo_produto,resgate_total_informado,resgate_parcial_informado,resgate_programado_informado,count
202004,05142,VGBL,0,0,0,10
201911,05142,VGBL,0,0,0,10
202001,05142,VGBL,0,0,0,10
201909,05142,VGBL,0,0,0,10
202102,04251,VGBL,0,0,0,10
201907,05142,VGBL,0,0,0,10
201905,05142,VGBL,0,0,0,10
201912,05142,VGBL,0,0,0,10
202007,05142,VGBL,0,0,0,10
201910,05142,VGBL,0,0,0,10


In [0]:
#confrmando separadamente por produto
display(
    df_grupos_duplicados_resgates
    .groupBy("tipo_produto")
    .agg(
        F.count("*").alias("grupos_duplicados"),
        F.sum(F.col("count") - 1).alias("linhas_excedentes"),
        F.max("count").alias("maior_repeticao")
    )
)

tipo_produto,grupos_duplicados,linhas_excedentes,maior_repeticao
PGBL,2898,6451,8
VGBL,3020,7808,10


Na base de resgates foram encontrados grupos de linhas com conteúdo idêntico, muitos deles com valores zerados. Como o arquivo público não contém identificador de plano, contrato ou outro campo capaz de distinguir esses registros, não foi possível afirmar que sejam duplicatas indevidas.

Para preservar a fidelidade à fonte, nenhuma linha foi excluída. Foi adicionada uma flag e a quantidade de ocorrências de cada combinação. A agregação será realizada somente na camada Gold, de acordo com a granularidade definida para a análise.

In [0]:
from pyspark.sql.window import Window

# Identificação de registros repetidos sem excluí-los

janela_repeticao_resgates = Window.partitionBy(
    "ano_mes",
    "codigo_fip",
    "tipo_produto",
    "resgate_total_informado",
    "resgate_parcial_informado",
    "resgate_programado_informado"
)

df_susep_resgates_silver = (
    df_susep_resgates_silver
    .withColumn(
        "quantidade_ocorrencias_fonte",
        F.count("*").over(janela_repeticao_resgates)
    )
    .withColumn(
        "flag_conteudo_repetido",
        F.col("quantidade_ocorrencias_fonte") > 1
    )
)

In [0]:
# Gravação das tabelas de fundos e resgates

(
    df_susep_fundos_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.susep_fundos")
)

(
    df_susep_resgates_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.susep_resgates")
)

print("Tabelas SUSEP de fundos e resgates gravadas com sucesso.")

Tabelas SUSEP de fundos e resgates gravadas com sucesso.


In [0]:
#validando os dados gravados
df_fundos_gravado = spark.table(
    "workspace.silver.susep_fundos"
)

df_resgates_gravado = spark.table(
    "workspace.silver.susep_resgates"
)

display(
    spark.createDataFrame(
        [
            ("susep_fundos", df_fundos_gravado.count()),
            ("susep_resgates", df_resgates_gravado.count())
        ],
        ["tabela", "quantidade_registros"]
    )
)

display(
    df_resgates_gravado.agg(
        F.sum(
            F.when(F.col("flag_conteudo_repetido"), 1).otherwise(0)
        ).alias("registros_em_grupos_repetidos"),
        F.max("quantidade_ocorrencias_fonte")
            .alias("maior_quantidade_ocorrencias")
    )
)

tabela,quantidade_registros
susep_fundos,12992
susep_resgates,36041


registros_em_grupos_repetidos,maior_quantidade_ocorrencias
20177,10


### 3.3 Informações de previdência aberta por UF

As bases de PGBL e previdência tradicional por unidade da Federação foram consolidadas em uma única tabela, com uma coluna para identificação do produto.

A camada Silver preserva todas as UFs disponíveis. O recorte da Região Sudeste será aplicado na camada Gold. Valores monetários foram convertidos para decimal e quantidades para números inteiros, mantendo os campos originais para rastreabilidade.

In [0]:
# Função para tratamento das bases SUSEP por UF

def preparar_susep_uf(nome_tabela, produto):
    df = spark.table(f"workspace.bronze.{nome_tabela}")

    return (
        df
        .select(
            F.lpad(F.trim(F.col("COENTI")), 5, "0").alias("codigo_fip"),
            F.trim(F.col("DAMESANO")).alias("ano_mes"),
            F.upper(F.trim(F.col("UF"))).alias("uf"),

            F.trim(F.col("CONTRIB")).alias("contribuicao_informada"),
            F.trim(F.col("BENEFPAGO")).alias("beneficio_pago_informado"),
            F.trim(F.col("RESGPAGO")).alias("resgate_pago_informado"),

            F.trim(F.col("NUMPARTIC")).alias("participantes_informado"),
            F.trim(F.col("NUMBENEF")).alias("beneficiarios_informado"),
            F.trim(F.col("NUMRESG")).alias("resgates_informado"),

            F.lit(produto).alias("tipo_produto"),
            F.col("_arquivo_origem"),
            F.col("_data_ingestao")
        )
        .withColumn(
            "data_referencia",
            F.to_date(
                F.concat(F.col("ano_mes"), F.lit("01")),
                "yyyyMMdd"
            )
        )
        .withColumn(
            "contribuicao",
            F.regexp_replace(
                F.col("contribuicao_informada"), ",", "."
            ).cast(tipo_monetario)
        )
        .withColumn(
            "beneficio_pago",
            F.regexp_replace(
                F.col("beneficio_pago_informado"), ",", "."
            ).cast(tipo_monetario)
        )
        .withColumn(
            "resgate_pago",
            F.regexp_replace(
                F.col("resgate_pago_informado"), ",", "."
            ).cast(tipo_monetario)
        )
        .withColumn(
            "participantes_original",
            F.col("participantes_informado").cast("long")
        )
        .withColumn(
            "beneficiarios_original",
            F.col("beneficiarios_informado").cast("long")
        )
        .withColumn(
            "resgates_original",
            F.col("resgates_informado").cast("long")
        )
        .withColumn(
            "participantes",
            F.when(
                F.col("participantes_original") >= 0,
                F.col("participantes_original")
            ).otherwise(F.lit(None).cast("long"))
        )
        .withColumn(
            "beneficiarios",
            F.when(
                F.col("beneficiarios_original") >= 0,
                F.col("beneficiarios_original")
            ).otherwise(F.lit(None).cast("long"))
        )
        .withColumn(
            "quantidade_resgates",
            F.when(
                F.col("resgates_original") >= 0,
                F.col("resgates_original")
            ).otherwise(F.lit(None).cast("long"))
        )
        .filter(F.col("ano_mes") <= "202512")
        .withColumn(
            "flag_valor_monetario_negativo",
            (F.col("contribuicao") < 0)
            | (F.col("beneficio_pago") < 0)
            | (F.col("resgate_pago") < 0)
        )
        .withColumn(
            "flag_quantidade_negativa",
            (F.col("participantes_original") < 0)
            | (F.col("beneficiarios_original") < 0)
            | (F.col("resgates_original") < 0)
        )
        .withColumn(
            "flag_entidade_sem_cadastro_atual",
            ~F.col("codigo_fip").isin(
                [
                    linha["codigo_fip"]
                    for linha in df_susep_empresas_silver
                    .select("codigo_fip")
                    .distinct()
                    .collect()
                ]
            )
        )
    )

In [0]:
df_susep_uf_silver = (
    preparar_susep_uf("susep_pgbl_uf", "PGBL")
    .unionByName(
        preparar_susep_uf("susep_prev_uf", "PREVTRAD")
    )
)

display(df_susep_uf_silver.limit(20))

codigo_fip,ano_mes,uf,contribuicao_informada,beneficio_pago_informado,resgate_pago_informado,participantes_informado,beneficiarios_informado,resgates_informado,tipo_produto,_arquivo_origem,_data_ingestao,data_referencia,contribuicao,beneficio_pago,resgate_pago,participantes_original,beneficiarios_original,resgates_original,participantes,beneficiarios,quantidade_resgates,flag_valor_monetario_negativo,flag_quantidade_negativa,flag_entidade_sem_cadastro_atual
06866,201305,MA,"355930,81",null,"752214,35",690,null,6,PGBL,ses_pgbl_uf.csv,2026-09-07T03:37:49.525Z,2013-05-01,355930.81,null,752214.35,690,null,6,690,null,6,null,null,false
06866,201305,MG,"2808069,65",null,"2103848,56",4400,null,50,PGBL,ses_pgbl_uf.csv,2026-09-07T03:37:49.525Z,2013-05-01,2808069.65,null,2103848.56,4400,null,50,4400,null,50,null,null,false
06866,201305,MS,"232212,06",null,"221494,52",937,null,3,PGBL,ses_pgbl_uf.csv,2026-09-07T03:37:49.525Z,2013-05-01,232212.06,null,221494.52,937,null,3,937,null,3,null,null,false
06866,201305,MT,"253112,79",null,"147047,13",818,null,4,PGBL,ses_pgbl_uf.csv,2026-09-07T03:37:49.525Z,2013-05-01,253112.79,null,147047.13,818,null,4,818,null,4,null,null,false
06866,201305,PA,"773007,61",null,"215938,49",1349,null,7,PGBL,ses_pgbl_uf.csv,2026-09-07T03:37:49.525Z,2013-05-01,773007.61,null,215938.49,1349,null,7,1349,null,7,null,null,false
06866,201305,PB,"601934,56",null,"491955,49",973,null,10,PGBL,ses_pgbl_uf.csv,2026-09-07T03:37:49.525Z,2013-05-01,601934.56,null,491955.49,973,null,10,973,null,10,null,null,false
06866,201305,PE,"973927,47",null,"152493,79",1458,null,11,PGBL,ses_pgbl_uf.csv,2026-09-07T03:37:49.525Z,2013-05-01,973927.47,null,152493.79,1458,null,11,1458,null,11,null,null,false
06866,201305,PR,"1392301,19",null,"1635463,8",2510,null,32,PGBL,ses_pgbl_uf.csv,2026-09-07T03:37:49.525Z,2013-05-01,1392301.19,null,1635463.80,2510,null,32,2510,null,32,null,null,false
06866,201305,RJ,"5572141,36",null,"1860606,15",9601,null,79,PGBL,ses_pgbl_uf.csv,2026-09-07T03:37:49.525Z,2013-05-01,5572141.36,null,1860606.15,9601,null,79,9601,null,79,null,null,false
06866,201305,RO,"141704,1",null,"5422,43",470,null,0,PGBL,ses_pgbl_uf.csv,2026-09-07T03:37:49.525Z,2013-05-01,141704.10,null,5422.43,470,null,0,470,null,0,null,null,false


In [0]:
# Validação da base SUSEP por UF

ufs_validas = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF",
    "ES", "GO", "MA", "MT", "MS", "MG", "PA",
    "PB", "PR", "PE", "PI", "RJ", "RN", "RS",
    "RO", "RR", "SC", "SP", "SE", "TO"
]

df_validacao_susep_uf = (
    df_susep_uf_silver
    .agg(
        F.count("*").alias("registros"),
        F.min("ano_mes").alias("primeiro_periodo"),
        F.max("ano_mes").alias("ultimo_periodo"),
        F.countDistinct("uf").alias("ufs_distintas"),
        F.sum(
            F.when(~F.col("uf").isin(ufs_validas), 1).otherwise(0)
        ).alias("ufs_invalidas"),
        F.sum(
            F.when(F.col("data_referencia").isNull(), 1).otherwise(0)
        ).alias("datas_invalidas"),
        F.sum(
            F.when(
                F.col("flag_valor_monetario_negativo"), 1
            ).otherwise(0)
        ).alias("valores_monetarios_negativos"),
        F.sum(
            F.when(F.col("flag_quantidade_negativa"), 1).otherwise(0)
        ).alias("quantidades_negativas"),
        F.sum(
            F.when(
                F.col("flag_entidade_sem_cadastro_atual"), 1
            ).otherwise(0)
        ).alias("registros_sem_cadastro_atual")
    )
)

display(df_validacao_susep_uf)

registros,primeiro_periodo,ultimo_periodo,ufs_distintas,ufs_invalidas,datas_invalidas,valores_monetarios_negativos,quantidades_negativas,registros_sem_cadastro_atual
406034,200101,202512,27,0,0,1112,56,76290


In [0]:
#conferindo especificamente a região Sudeste
display(
    df_susep_uf_silver
    .filter(F.col("uf").isin(["MG", "ES", "RJ", "SP"]))
    .groupBy("tipo_produto", "uf")
    .agg(
        F.count("*").alias("registros"),
        F.min("ano_mes").alias("primeiro_periodo"),
        F.max("ano_mes").alias("ultimo_periodo")
    )
    .orderBy("tipo_produto", "uf")
)

tipo_produto,uf,registros,primeiro_periodo,ultimo_periodo
PGBL,ES,5425,200101,202512
PGBL,MG,5830,200101,202512
PGBL,RJ,6155,200101,202512
PGBL,SP,6618,200101,202512
PREVTRAD,ES,10680,200101,202512
PREVTRAD,MG,11851,200101,202512
PREVTRAD,RJ,13121,200101,202512
PREVTRAD,SP,13083,200101,202512


In [0]:
# Correção das flags e validação das conversões

df_susep_uf_silver = (
    df_susep_uf_silver
    .withColumn(
        "flag_valor_monetario_negativo",
        F.coalesce(
            (F.col("contribuicao") < 0)
            | (F.col("beneficio_pago") < 0)
            | (F.col("resgate_pago") < 0),
            F.lit(False)
        )
    )
    .withColumn(
        "flag_quantidade_negativa",
        F.coalesce(
            (F.col("participantes_original") < 0)
            | (F.col("beneficiarios_original") < 0)
            | (F.col("resgates_original") < 0),
            F.lit(False)
        )
    )
    .withColumn(
        "flag_valor_nao_convertido",
        (
            F.col("contribuicao_informada").isNotNull()
            & (F.trim(F.col("contribuicao_informada")) != "")
            & F.col("contribuicao").isNull()
        )
        | (
            F.col("beneficio_pago_informado").isNotNull()
            & (F.trim(F.col("beneficio_pago_informado")) != "")
            & F.col("beneficio_pago").isNull()
        )
        | (
            F.col("resgate_pago_informado").isNotNull()
            & (F.trim(F.col("resgate_pago_informado")) != "")
            & F.col("resgate_pago").isNull()
        )
    )
    .withColumn(
        "flag_quantidade_nao_convertida",
        (
            F.col("participantes_informado").isNotNull()
            & (F.trim(F.col("participantes_informado")) != "")
            & F.col("participantes_original").isNull()
        )
        | (
            F.col("beneficiarios_informado").isNotNull()
            & (F.trim(F.col("beneficiarios_informado")) != "")
            & F.col("beneficiarios_original").isNull()
        )
        | (
            F.col("resgates_informado").isNotNull()
            & (F.trim(F.col("resgates_informado")) != "")
            & F.col("resgates_original").isNull()
        )
    )
)

In [0]:
#examinando os registros com quantidades negativas
display(
    df_susep_uf_silver
    .filter(F.col("flag_quantidade_negativa"))
    .select(
        "ano_mes",
        "codigo_fip",
        "tipo_produto",
        "uf",
        "participantes",
        "participantes_original",
        "beneficiarios",
        "beneficiarios_original",
        "quantidade_resgates",
        "resgates_original"
    )
)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7294493473844009>, line 3
      1 #examinando os registros com quantidades negativas
      2 display(
----> 3     df_susep_uf_silver
      4     .filter(F.col("flag_quantidade_negativa"))
      5     .select(
      6         "ano_mes",
      7         "codigo_fip",
      8         "tipo_produto",
      9         "uf",
     10         "participantes",
     11         "participantes_original",
     12         "beneficiarios",
     13         "beneficiarios_original",
     14         "quantidade_resgates",
     15         "resgates_original"
     16     )
     17 )

NameError: name 'df_susep_uf_silver' is not defined

In [0]:
#validando as conversões
display(
    df_susep_uf_silver.agg(
        F.count("*").alias("registros"),
        F.sum(
            F.when(F.col("flag_valor_nao_convertido"), 1).otherwise(0)
        ).alias("valores_nao_convertidos"),
        F.sum(
            F.when(
                F.col("flag_quantidade_nao_convertida"), 1
            ).otherwise(0)
        ).alias("quantidades_nao_convertidas"),
        F.sum(
            F.when(
                F.col("flag_valor_monetario_negativo"), 1
            ).otherwise(0)
        ).alias("valores_monetarios_negativos"),
        F.sum(
            F.when(F.col("flag_quantidade_negativa"), 1).otherwise(0)
        ).alias("quantidades_negativas"),
        F.sum(
            F.when(
                F.col("flag_entidade_sem_cadastro_atual"), 1
            ).otherwise(0)
        ).alias("registros_sem_cadastro_atual")
    )
)

registros,valores_nao_convertidos,quantidades_nao_convertidas,valores_monetarios_negativos,quantidades_negativas,registros_sem_cadastro_atual
406034,0,0,1112,56,76290


In [0]:
# Diagnóstico final das quantidades e conversões da base por UF

df_diagnostico_final_uf = df_susep_uf_silver.agg(
    F.count("*").alias("registros"),

    F.sum(
        F.when(F.col("participantes_original") < 0, 1).otherwise(0)
    ).alias("participantes_negativos"),

    F.sum(
        F.when(F.col("beneficiarios_original") < 0, 1).otherwise(0)
    ).alias("beneficiarios_negativos"),

    F.sum(
        F.when(F.col("resgates_original") < 0, 1).otherwise(0)
    ).alias("quantidades_resgates_negativas"),

    F.sum(
        F.when(F.col("flag_valor_nao_convertido"), 1).otherwise(0)
    ).alias("valores_monetarios_nao_convertidos"),

    F.sum(
        F.when(
            F.col("flag_quantidade_nao_convertida"), 1
        ).otherwise(0)
    ).alias("quantidades_nao_convertidas"),

    F.sum(
        F.when(
            F.col("flag_valor_monetario_negativo"), 1
        ).otherwise(0)
    ).alias("registros_monetarios_negativos"),

    F.sum(
        F.when(F.col("flag_quantidade_negativa"), 1).otherwise(0)
    ).alias("registros_quantidade_negativa")
)

display(df_diagnostico_final_uf)

registros,participantes_negativos,beneficiarios_negativos,quantidades_resgates_negativas,valores_monetarios_nao_convertidos,quantidades_nao_convertidas,registros_monetarios_negativos,registros_quantidade_negativa
406034,16,26,14,0,0,1112,56


In [0]:
#visualizando os registros
display(
    df_susep_uf_silver
    .filter(F.col("flag_quantidade_negativa"))
    .select(
        "ano_mes",
        "codigo_fip",
        "tipo_produto",
        "uf",
        "participantes",
        "participantes_original",
        "beneficiarios",
        "beneficiarios_original",
        "quantidade_resgates",
        "resgates_original"
    )
    .orderBy("ano_mes", "codigo_fip", "uf")
)

ano_mes,codigo_fip,tipo_produto,uf,participantes,participantes_original,beneficiarios,beneficiarios_original,quantidade_resgates,resgates_original
200107,05215,PREVTRAD,GO,3484,3484,0,0,null,-9
200107,05215,PREVTRAD,MG,16915,16915,21,21,null,-99
200107,05215,PREVTRAD,RJ,46884,46884,164,164,null,-1646
201009,10138,PREVTRAD,TO,null,-66,0,0,0,0
201607,02101,PREVTRAD,MG,null,-458093,163,163,2,2
201607,02101,PREVTRAD,MS,null,-400274,19,19,0,0
201607,02101,PREVTRAD,PA,null,-13158,33,33,0,0
201707,03298,PGBL,GO,83,83,null,null,null,-2
201906,03298,PGBL,TO,12,12,null,null,null,-1
202004,03298,PGBL,MT,43,43,null,null,null,-1


In [0]:
# Verificação de conteúdo repetido na base SUSEP por UF

colunas_conteudo_uf = [
    "ano_mes",
    "codigo_fip",
    "tipo_produto",
    "uf",
    "contribuicao_informada",
    "beneficio_pago_informado",
    "resgate_pago_informado",
    "participantes_informado",
    "beneficiarios_informado",
    "resgates_informado"
]

df_repeticoes_susep_uf = (
    df_susep_uf_silver
    .groupBy(*colunas_conteudo_uf)
    .count()
    .filter(F.col("count") > 1)
)

display(
    df_repeticoes_susep_uf.agg(
        F.count("*").alias("grupos_repetidos"),
        F.sum(F.col("count") - 1).alias("linhas_excedentes"),
        F.max("count").alias("maior_repeticao")
    )
)

grupos_repetidos,linhas_excedentes,maior_repeticao
0,null,null


In [0]:
# Gravação da tabela SUSEP por UF

(
    df_susep_uf_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.susep_previdencia_uf")
)

print("Tabela workspace.silver.susep_previdencia_uf gravada com sucesso.")

Tabela workspace.silver.susep_previdencia_uf gravada com sucesso.


In [0]:
#validando a gravação
df_susep_uf_gravado = spark.table(
    "workspace.silver.susep_previdencia_uf"
)

display(
    df_susep_uf_gravado.agg(
        F.count("*").alias("quantidade_registros"),
        F.countDistinct("uf").alias("quantidade_ufs"),
        F.countDistinct("codigo_fip").alias("quantidade_entidades"),
        F.min("ano_mes").alias("primeiro_periodo"),
        F.max("ano_mes").alias("ultimo_periodo"),
        F.sum(
            F.when(F.col("flag_quantidade_negativa"), 1).otherwise(0)
        ).alias("quantidades_negativas_sinalizadas"),
        F.sum(
            F.when(F.col("data_referencia").isNull(), 1).otherwise(0)
        ).alias("datas_invalidas")
    )
)

quantidade_registros,quantidade_ufs,quantidade_entidades,primeiro_periodo,ultimo_periodo,quantidades_negativas_sinalizadas,datas_invalidas
406034,27,128,200101,202512,56,0


### 3.4 Inclusão dos resgates de previdência tradicional

A base de resgates de previdência tradicional possui estrutura semelhante às bases de PGBL e VGBL, mas não apresenta a variável de resgate por pagamento programado. Esse campo foi mantido como nulo para PREVTRAD, permitindo a consolidação dos três produtos em uma única tabela.

In [0]:
# Recuperação das importações e DataFrames após reinício da sessão

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType
from pyspark.sql.window import Window

tipo_monetario = DecimalType(20, 2)

# Recupera tabelas já gravadas
df_susep_empresas_silver = spark.table(
    "workspace.silver.susep_lista_empresas"
)

df_susep_resgates_silver = spark.table(
    "workspace.silver.susep_resgates"
)

# Recria a referência dos códigos cadastrados
df_codigos_susep_cadastrados = (
    df_susep_empresas_silver
    .select("codigo_fip")
    .distinct()
    .withColumn("consta_cadastro_atual", F.lit(True))
)

print("Ambiente recuperado com sucesso.")

Ambiente recuperado com sucesso.


In [0]:
# Tratamento dos resgates de previdência tradicional

df_prevtrad_resgates_bronze = spark.table(
    "workspace.bronze.susep_prev_trad_resgates"
)

df_prevtrad_resgates_silver = (
    df_prevtrad_resgates_bronze
    .select(
        F.lpad(F.trim(F.col("coenti")), 5, "0").alias("codigo_fip"),
        F.trim(F.col("damesano")).alias("ano_mes"),
        F.trim(F.col("resg_total"))
            .alias("resgate_total_informado"),
        F.trim(F.col("resg_parcial"))
            .alias("resgate_parcial_informado"),
        F.lit(None).cast("string")
            .alias("resgate_programado_informado"),
        F.lit("PREVTRAD").alias("tipo_produto"),
        F.col("_arquivo_origem"),
        F.col("_data_ingestao")
    )
    .withColumn(
        "data_referencia",
        F.to_date(
            F.concat(F.col("ano_mes"), F.lit("01")),
            "yyyyMMdd"
        )
    )
    .withColumn(
        "resgate_total",
        F.regexp_replace(
            F.col("resgate_total_informado"), ",", "."
        ).cast(tipo_monetario)
    )
    .withColumn(
        "resgate_parcial",
        F.regexp_replace(
            F.col("resgate_parcial_informado"), ",", "."
        ).cast(tipo_monetario)
    )
    .withColumn(
        "resgate_programado",
        F.lit(None).cast(tipo_monetario)
    )
    .filter(F.col("ano_mes") <= "202512")
    .withColumn(
        "flag_valor_nao_convertido",
        (
            F.col("resgate_total_informado").isNotNull()
            & (F.trim(F.col("resgate_total_informado")) != "")
            & F.col("resgate_total").isNull()
        )
        | (
            F.col("resgate_parcial_informado").isNotNull()
            & (F.trim(F.col("resgate_parcial_informado")) != "")
            & F.col("resgate_parcial").isNull()
        )
    )
    .withColumn(
        "flag_valor_negativo",
        F.coalesce(
            (F.col("resgate_total") < 0)
            | (F.col("resgate_parcial") < 0),
            F.lit(False)
        )
    )
    .join(
        df_codigos_susep_cadastrados,
        on="codigo_fip",
        how="left"
    )
    .withColumn(
        "flag_entidade_sem_cadastro_atual",
        F.col("consta_cadastro_atual").isNull()
    )
    .drop("consta_cadastro_atual")
)

In [0]:
# Consolidação de PGBL, VGBL e PREVTRAD

df_resgates_pgbl_vgbl = (
    df_susep_resgates_silver
    .drop(
        "quantidade_ocorrencias_fonte",
        "flag_conteudo_repetido"
    )
)

df_susep_resgates_completo = (
    df_resgates_pgbl_vgbl
    .unionByName(
        df_prevtrad_resgates_silver,
        allowMissingColumns=True
    )
)

janela_repeticao_resgates_completa = Window.partitionBy(
    "ano_mes",
    "codigo_fip",
    "tipo_produto",
    "resgate_total_informado",
    "resgate_parcial_informado",
    "resgate_programado_informado"
)

df_susep_resgates_completo = (
    df_susep_resgates_completo
    .withColumn(
        "quantidade_ocorrencias_fonte",
        F.count("*").over(
            janela_repeticao_resgates_completa
        )
    )
    .withColumn(
        "flag_conteudo_repetido",
        F.col("quantidade_ocorrencias_fonte") > 1
    )
)

display(df_susep_resgates_completo.limit(20))

codigo_fip,ano_mes,resgate_total_informado,resgate_parcial_informado,resgate_programado_informado,tipo_produto,_arquivo_origem,_data_ingestao,data_referencia,resgate_total,resgate_parcial,resgate_programado,flag_valor_nao_convertido,flag_valor_negativo,flag_entidade_sem_cadastro_atual,quantidade_ocorrencias_fonte,flag_conteudo_repetido
01007,200401,"555,42",0,null,PREVTRAD,ses_prev_trad_resgates.csv,2026-09-07T03:37:54.347Z,2004-01-01,555.42,0.00,null,false,false,false,1,false
03816,200401,"1606899,41","5747944,07",0,PGBL,ses_pgbl_resgates.csv,2026-09-07T03:37:45.170Z,2004-01-01,1606899.41,5747944.07,0.00,false,false,true,1,false
04707,200401,"5831003,21","5459024,59",0,PGBL,ses_pgbl_resgates.csv,2026-09-07T03:37:45.170Z,2004-01-01,5831003.21,5459024.59,0.00,false,false,false,1,false
04707,200401,"11477969,2","13451530,92",null,PREVTRAD,ses_prev_trad_resgates.csv,2026-09-07T03:37:54.347Z,2004-01-01,11477969.20,13451530.92,null,false,false,false,1,false
05142,200401,"217601,14","3706147,17",0,PGBL,ses_pgbl_resgates.csv,2026-09-07T03:37:45.170Z,2004-01-01,217601.14,3706147.17,0.00,false,false,false,1,false
05193,200401,191378,0,null,PREVTRAD,ses_prev_trad_resgates.csv,2026-09-07T03:37:54.347Z,2004-01-01,191378.00,0.00,null,false,false,false,1,false
05215,200401,"7205437,85","9488159,04",0,PGBL,ses_pgbl_resgates.csv,2026-09-07T03:37:45.170Z,2004-01-01,7205437.85,9488159.04,0.00,false,false,true,1,false
05215,200401,"2978133,27","11682746,9",null,PREVTRAD,ses_prev_trad_resgates.csv,2026-09-07T03:37:54.347Z,2004-01-01,2978133.27,11682746.90,null,false,false,true,1,false
05789,200401,"43346,75",0,null,PREVTRAD,ses_prev_trad_resgates.csv,2026-09-07T03:37:54.347Z,2004-01-01,43346.75,0.00,null,false,false,true,1,false
06033,200401,"192073,6","138835,37",0,PGBL,ses_pgbl_resgates.csv,2026-09-07T03:37:45.170Z,2004-01-01,192073.60,138835.37,0.00,false,false,false,1,false


In [0]:
# Validação dos resgates completos

display(
    df_susep_resgates_completo
    .groupBy("tipo_produto")
    .agg(
        F.count("*").alias("registros"),
        F.min("ano_mes").alias("primeiro_periodo"),
        F.max("ano_mes").alias("ultimo_periodo"),
        F.sum(
            F.when(F.col("flag_valor_nao_convertido"), 1).otherwise(0)
        ).alias("valores_nao_convertidos"),
        F.sum(
            F.when(F.col("flag_valor_negativo"), 1).otherwise(0)
        ).alias("valores_negativos"),
        F.sum(
            F.when(F.col("flag_conteudo_repetido"), 1).otherwise(0)
        ).alias("registros_em_grupos_repetidos")
    )
    .orderBy("tipo_produto")
)

tipo_produto,registros,primeiro_periodo,ultimo_periodo,valores_nao_convertidos,valores_negativos,registros_em_grupos_repetidos
PGBL,17250,200401,202512,0,0,9349
PREVTRAD,19499,200401,202512,0,3,11308
VGBL,18791,200401,202512,0,0,10828


In [0]:
# Gravação da versão completa dos resgates

(
    df_susep_resgates_completo.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.susep_resgates")
)

print("Tabela completa workspace.silver.susep_resgates gravada.")

Tabela completa workspace.silver.susep_resgates gravada.


In [0]:
#confirmando que a gravação preservou tudo
df_resgates_gravado = spark.table(
    "workspace.silver.susep_resgates"
)

display(
    df_resgates_gravado.agg(
        F.count("*").alias("quantidade_registros"),
        F.countDistinct("tipo_produto").alias("tipos_produto"),
        F.min("ano_mes").alias("primeiro_periodo"),
        F.max("ano_mes").alias("ultimo_periodo"),
        F.sum(
            F.when(F.col("flag_valor_nao_convertido"), 1).otherwise(0)
        ).alias("valores_nao_convertidos"),
        F.sum(
            F.when(F.col("flag_valor_negativo"), 1).otherwise(0)
        ).alias("valores_negativos")
    )
)

quantidade_registros,tipos_produto,primeiro_periodo,ultimo_periodo,valores_nao_convertidos,valores_negativos
55540,3,200401,202512,0,3


### 3.5 Provisões e movimentação de participantes e beneficiários

A base de provisões contém valores monetários relacionados a seguros e previdência. As bases de participantes e beneficiários apresentam estoques, inclusões, exclusões e, no caso dos participantes, cancelamentos.

As duas bases populacionais foram consolidadas, preservando a identificação do grupo. Quantidades negativas são mantidas nos campos originais, transformadas em nulo nos campos analíticos e sinalizadas por flags.

In [0]:
# Tratamento da base de provisões

df_provisoes_bronze = spark.table(
    "workspace.bronze.susep_prov_segprev"
)

df_susep_provisoes_silver = (
    df_provisoes_bronze
    .select(
        F.lpad(F.trim(F.col("coenti")), 5, "0").alias("codigo_fip"),
        F.trim(F.col("damesano")).alias("ano_mes"),
        F.trim(F.col("valor")).alias("valor_provisao_informado"),
        F.col("_arquivo_origem"),
        F.col("_data_ingestao")
    )
    .withColumn(
        "data_referencia",
        F.to_date(
            F.concat(F.col("ano_mes"), F.lit("01")),
            "yyyyMMdd"
        )
    )
    .withColumn(
        "valor_provisao",
        F.regexp_replace(
            F.col("valor_provisao_informado"), ",", "."
        ).cast(tipo_monetario)
    )
    .filter(F.col("ano_mes") <= "202512")
    .withColumn(
        "flag_valor_nao_convertido",
        F.col("valor_provisao_informado").isNotNull()
        & (F.trim(F.col("valor_provisao_informado")) != "")
        & F.col("valor_provisao").isNull()
    )
    .withColumn(
        "flag_valor_negativo",
        F.coalesce(
            F.col("valor_provisao") < 0,
            F.lit(False)
        )
    )
    .join(
        df_codigos_susep_cadastrados,
        on="codigo_fip",
        how="left"
    )
    .withColumn(
        "flag_entidade_sem_cadastro_atual",
        F.col("consta_cadastro_atual").isNull()
    )
    .drop("consta_cadastro_atual")
)

In [0]:
# Função de tratamento de participantes e beneficiários

def preparar_quantidades_susep(
    nome_tabela,
    grupo_populacional,
    possui_cancelados
):
    df = spark.table(f"workspace.bronze.{nome_tabela}")

    if possui_cancelados:
        coluna_cancelados = F.trim(
            F.col("CANCELADOS")
        ).alias("cancelados_informado")
    else:
        coluna_cancelados = F.lit(None).cast(
            "string"
        ).alias("cancelados_informado")

    return (
        df
        .select(
            F.lpad(F.trim(F.col("COENTI")), 5, "0").alias("codigo_fip"),
            F.trim(F.col("DAMESANO")).alias("ano_mes"),
            F.upper(F.trim(F.col("TIPO"))).alias("tipo_produto"),
            F.trim(F.col("ESTOQUE")).alias("estoque_informado"),
            F.trim(F.col("INCLUSOES")).alias("inclusoes_informado"),
            F.trim(F.col("EXCLUSOES")).alias("exclusoes_informado"),
            coluna_cancelados,
            F.lit(grupo_populacional).alias("grupo_populacional"),
            F.col("_arquivo_origem"),
            F.col("_data_ingestao")
        )
        .withColumn(
            "data_referencia",
            F.to_date(
                F.concat(F.col("ano_mes"), F.lit("01")),
                "yyyyMMdd"
            )
        )
        .withColumn(
            "estoque_original",
            F.col("estoque_informado").cast("long")
        )
        .withColumn(
            "inclusoes_original",
            F.col("inclusoes_informado").cast("long")
        )
        .withColumn(
            "exclusoes_original",
            F.col("exclusoes_informado").cast("long")
        )
        .withColumn(
            "cancelados_original",
            F.col("cancelados_informado").cast("long")
        )
        .withColumn(
            "estoque",
            F.when(
                F.col("estoque_original") >= 0,
                F.col("estoque_original")
            ).otherwise(F.lit(None).cast("long"))
        )
        .withColumn(
            "inclusoes",
            F.when(
                F.col("inclusoes_original") >= 0,
                F.col("inclusoes_original")
            ).otherwise(F.lit(None).cast("long"))
        )
        .withColumn(
            "exclusoes",
            F.when(
                F.col("exclusoes_original") >= 0,
                F.col("exclusoes_original")
            ).otherwise(F.lit(None).cast("long"))
        )
        .withColumn(
            "cancelados",
            F.when(
                F.col("cancelados_original") >= 0,
                F.col("cancelados_original")
            ).otherwise(F.lit(None).cast("long"))
        )
        .filter(F.col("ano_mes") <= "202512")
        .withColumn(
            "flag_quantidade_negativa",
            F.coalesce(
                (F.col("estoque_original") < 0)
                | (F.col("inclusoes_original") < 0)
                | (F.col("exclusoes_original") < 0)
                | (F.col("cancelados_original") < 0),
                F.lit(False)
            )
        )
        .join(
            df_codigos_susep_cadastrados,
            on="codigo_fip",
            how="left"
        )
        .withColumn(
            "flag_entidade_sem_cadastro_atual",
            F.col("consta_cadastro_atual").isNull()
        )
        .drop("consta_cadastro_atual")
    )

In [0]:
#consolidando o tratamento de participantes e beneficiários 
df_susep_quantidades_silver = (
    preparar_quantidades_susep(
        "susep_quantprev_benef",
        "BENEFICIARIOS",
        False
    )
    .unionByName(
        preparar_quantidades_susep(
            "susep_quantprev_part",
            "PARTICIPANTES",
            True
        )
    )
)

display(df_susep_quantidades_silver.limit(20))

codigo_fip,ano_mes,tipo_produto,estoque_informado,inclusoes_informado,exclusoes_informado,cancelados_informado,grupo_populacional,_arquivo_origem,_data_ingestao,data_referencia,estoque_original,inclusoes_original,exclusoes_original,cancelados_original,estoque,inclusoes,exclusoes,cancelados,flag_quantidade_negativa,flag_entidade_sem_cadastro_atual
10031,200212,TRAD-APOSENTADORIA,4,0,0,null,BENEFICIARIOS,ses_quantprev_benef.csv,2026-09-07T03:38:08.260Z,2002-12-01,4,0,0,null,4,0,0,null,false,false
01007,201302,TRAD-APOSENTADORIA,39,0,0,null,BENEFICIARIOS,ses_quantprev_benef.csv,2026-09-07T03:38:08.260Z,2013-02-01,39,0,0,null,39,0,0,null,false,false
10065,201306,TRAD-APOSENTADORIA,3796,312,1,null,BENEFICIARIOS,ses_quantprev_benef.csv,2026-09-07T03:38:08.260Z,2013-06-01,3796,312,1,null,3796,312,1,null,false,true
10065,200805,TRAD-APOSENTADORIA,6134,413,387,null,BENEFICIARIOS,ses_quantprev_benef.csv,2026-09-07T03:38:08.260Z,2008-05-01,6134,413,387,null,6134,413,387,null,false,true
10235,200112,TRAD-APOSENTADORIA,24766,214,2,null,BENEFICIARIOS,ses_quantprev_benef.csv,2026-09-07T03:38:08.260Z,2001-12-01,24766,214,2,null,24766,214,2,null,false,true
06955,200208,TRAD-APOSENTADORIA,5,0,0,null,BENEFICIARIOS,ses_quantprev_benef.csv,2026-09-07T03:38:08.260Z,2002-08-01,5,0,0,null,5,0,0,null,false,true
05096,200811,TRAD-APOSENTADORIA,989,217,0,null,BENEFICIARIOS,ses_quantprev_benef.csv,2026-09-07T03:38:08.260Z,2008-11-01,989,217,0,null,989,217,0,null,false,false
06220,200207,TRAD-APOSENTADORIA,769,7,17,null,BENEFICIARIOS,ses_quantprev_benef.csv,2026-09-07T03:38:08.260Z,2002-07-01,769,7,17,null,769,7,17,null,false,false
06106,200110,TRAD-APOSENTADORIA,12,1,0,null,BENEFICIARIOS,ses_quantprev_benef.csv,2026-09-07T03:38:08.260Z,2001-10-01,12,1,0,null,12,1,0,null,false,true
05541,200009,TRAD-APOSENTADORIA,6,1,0,null,BENEFICIARIOS,ses_quantprev_benef.csv,2026-09-07T03:38:08.260Z,2000-09-01,6,1,0,null,6,1,0,null,false,true


In [0]:
#validando as duas estruturas de participantes e beneficiários
print("Validação das provisões:")

display(
    df_susep_provisoes_silver.agg(
        F.count("*").alias("registros"),
        F.min("ano_mes").alias("primeiro_periodo"),
        F.max("ano_mes").alias("ultimo_periodo"),
        F.sum(
            F.when(F.col("flag_valor_nao_convertido"), 1).otherwise(0)
        ).alias("valores_nao_convertidos"),
        F.sum(
            F.when(F.col("flag_valor_negativo"), 1).otherwise(0)
        ).alias("valores_negativos")
    )
)

print("Validação das quantidades:")

display(
    df_susep_quantidades_silver
    .groupBy("grupo_populacional")
    .agg(
        F.count("*").alias("registros"),
        F.min("ano_mes").alias("primeiro_periodo"),
        F.max("ano_mes").alias("ultimo_periodo"),
        F.countDistinct("tipo_produto").alias("tipos_produto"),
        F.sum(
            F.when(F.col("flag_quantidade_negativa"), 1).otherwise(0)
        ).alias("quantidades_negativas")
    )
    .orderBy("grupo_populacional")
)

Validação das provisões:


registros,primeiro_periodo,ultimo_periodo,valores_nao_convertidos,valores_negativos
8005,200301,202512,0,5


Validação das quantidades:


grupo_populacional,registros,primeiro_periodo,ultimo_periodo,tipos_produto,quantidades_negativas
BENEFICIARIOS,94054,199906,202512,6,9312
PARTICIPANTES,176941,199906,202512,6,23905


In [0]:
# Diagnóstico dos negativos por grupo populacional e campo

df_diagnostico_quantidades = (
    df_susep_quantidades_silver
    .groupBy("grupo_populacional")
    .agg(
        F.count("*").alias("registros"),

        F.sum(
            F.when(F.col("estoque_original") < 0, 1).otherwise(0)
        ).alias("estoques_negativos"),

        F.sum(
            F.when(F.col("inclusoes_original") < 0, 1).otherwise(0)
        ).alias("inclusoes_negativas"),

        F.sum(
            F.when(F.col("exclusoes_original") < 0, 1).otherwise(0)
        ).alias("exclusoes_negativas"),

        F.sum(
            F.when(F.col("cancelados_original") < 0, 1).otherwise(0)
        ).alias("cancelados_negativos"),

        F.min("estoque_original").alias("menor_estoque"),
        F.min("inclusoes_original").alias("menor_inclusao"),
        F.min("exclusoes_original").alias("menor_exclusao"),
        F.min("cancelados_original").alias("menor_cancelamento")
    )
)

display(df_diagnostico_quantidades)

grupo_populacional,registros,estoques_negativos,inclusoes_negativas,exclusoes_negativas,cancelados_negativos,menor_estoque,menor_inclusao,menor_exclusao,menor_cancelamento
BENEFICIARIOS,94054,9304,1,8,0,-110363,-2,-7,null
PARTICIPANTES,176941,23902,2,1,1,-1705563,-167967,-158,-5


In [0]:
#Analisando os negativos se distribuem entre os tipos e reistros
display(
    df_susep_quantidades_silver
    .filter(F.col("flag_quantidade_negativa"))
    .groupBy("grupo_populacional", "tipo_produto")
    .agg(
        F.count("*").alias("registros_negativos"),
        F.sum(
            F.when(F.col("estoque_original") < 0, 1).otherwise(0)
        ).alias("estoques_negativos"),
        F.sum(
            F.when(F.col("inclusoes_original") < 0, 1).otherwise(0)
        ).alias("inclusoes_negativas"),
        F.sum(
            F.when(F.col("exclusoes_original") < 0, 1).otherwise(0)
        ).alias("exclusoes_negativas"),
        F.sum(
            F.when(F.col("cancelados_original") < 0, 1).otherwise(0)
        ).alias("cancelados_negativos")
    )
    .orderBy("grupo_populacional", "tipo_produto")
)

grupo_populacional,tipo_produto,registros_negativos,estoques_negativos,inclusoes_negativas,exclusoes_negativas,cancelados_negativos
BENEFICIARIOS,PGBL,1269,1264,1,5,0
BENEFICIARIOS,TRAD-APOSENTADORIA,1736,1736,0,0,0
BENEFICIARIOS,TRAD-OUTROS,230,229,0,1,0
BENEFICIARIOS,TRAD-PECULIO,3390,3390,0,0,0
BENEFICIARIOS,TRAD-PENSAO,1774,1774,0,0,0
BENEFICIARIOS,VGBL,913,911,0,2,0
PARTICIPANTES,PGBL,4122,4122,0,0,0
PARTICIPANTES,TRAD-APOSENTADORIA,3853,3852,0,0,1
PARTICIPANTES,TRAD-OUTROS,1148,1148,0,0,0
PARTICIPANTES,TRAD-PECULIO,6822,6822,0,0,0


In [0]:
#analisando uma amostra dos valores negativos
display(
    df_susep_quantidades_silver
    .filter(F.col("flag_quantidade_negativa"))
    .select(
        "ano_mes",
        "codigo_fip",
        "grupo_populacional",
        "tipo_produto",
        "estoque_original",
        "inclusoes_original",
        "exclusoes_original",
        "cancelados_original"
    )
    .orderBy(
        "grupo_populacional",
        "tipo_produto",
        "ano_mes"
    )
    .limit(50)
)

ano_mes,codigo_fip,grupo_populacional,tipo_produto,estoque_original,inclusoes_original,exclusoes_original,cancelados_original
200105,06840,BENEFICIARIOS,PGBL,-129,2,132,null
200106,06840,BENEFICIARIOS,PGBL,-73,3,78,null
201312,05070,BENEFICIARIOS,PGBL,-385,0,385,null
201312,08141,BENEFICIARIOS,PGBL,-4,0,4,null
201312,04707,BENEFICIARIOS,PGBL,-204,0,204,null
201312,03298,BENEFICIARIOS,PGBL,-2,0,2,null
201312,04740,BENEFICIARIOS,PGBL,-3,0,3,null
201312,06033,BENEFICIARIOS,PGBL,-2,0,2,null
201312,05096,BENEFICIARIOS,PGBL,-105,0,105,null
201401,04707,BENEFICIARIOS,PGBL,-258,0,258,null


In [0]:
# Tratamento diferenciado entre estoque e movimentações

df_susep_quantidades_silver = (
    df_susep_quantidades_silver
    .withColumn(
        "estoque",
        F.when(
            F.col("estoque_original") >= 0,
            F.col("estoque_original")
        ).otherwise(F.lit(None).cast("long"))
    )
    # Movimentações negativas são preservadas como ajustes
    .withColumn("inclusoes", F.col("inclusoes_original"))
    .withColumn("exclusoes", F.col("exclusoes_original"))
    .withColumn("cancelados", F.col("cancelados_original"))
    .withColumn(
        "flag_estoque_negativo",
        F.coalesce(
            F.col("estoque_original") < 0,
            F.lit(False)
        )
    )
    .withColumn(
        "flag_movimentacao_negativa",
        F.coalesce(
            (F.col("inclusoes_original") < 0)
            | (F.col("exclusoes_original") < 0)
            | (F.col("cancelados_original") < 0),
            F.lit(False)
        )
    )
    .withColumn(
        "flag_quantidade_negativa",
        F.col("flag_estoque_negativo")
        | F.col("flag_movimentacao_negativa")
    )
    .withColumn(
        "flag_quantidade_nao_convertida",
        (
            F.col("estoque_informado").isNotNull()
            & (F.trim(F.col("estoque_informado")) != "")
            & F.col("estoque_original").isNull()
        )
        | (
            F.col("inclusoes_informado").isNotNull()
            & (F.trim(F.col("inclusoes_informado")) != "")
            & F.col("inclusoes_original").isNull()
        )
        | (
            F.col("exclusoes_informado").isNotNull()
            & (F.trim(F.col("exclusoes_informado")) != "")
            & F.col("exclusoes_original").isNull()
        )
        | (
            F.col("cancelados_informado").isNotNull()
            & (F.trim(F.col("cancelados_informado")) != "")
            & F.col("cancelados_original").isNull()
        )
    )
)

Os valores negativos estavam concentrados no campo de estoque. Como estoque representa uma quantidade de pessoas, valores negativos não possuem interpretação analítica e foram convertidos em nulos, permanecendo preservados no campo original.

Valores negativos de inclusões, exclusões e cancelamentos foram mantidos nos campos analíticos, pois podem representar ajustes ou estornos de movimentação. Foram criadas flags distintas para estoque negativo e movimentação negativa.

In [0]:
#validando os ajustes realizados
display(
    df_susep_quantidades_silver
    .groupBy("grupo_populacional")
    .agg(
        F.count("*").alias("registros"),
        F.sum(
            F.when(F.col("flag_estoque_negativo"), 1).otherwise(0)
        ).alias("estoques_negativos_tratados"),
        F.sum(
            F.when(
                F.col("flag_movimentacao_negativa"), 1
            ).otherwise(0)
        ).alias("movimentacoes_negativas_preservadas"),
        F.sum(
            F.when(
                F.col("flag_quantidade_nao_convertida"), 1
            ).otherwise(0)
        ).alias("quantidades_nao_convertidas"),
        F.sum(
            F.when(F.col("estoque") < 0, 1).otherwise(0)
        ).alias("estoques_negativos_no_campo_analitico")
    )
)

grupo_populacional,registros,estoques_negativos_tratados,movimentacoes_negativas_preservadas,quantidades_nao_convertidas,estoques_negativos_no_campo_analitico
BENEFICIARIOS,94054,9304,9,0,0
PARTICIPANTES,176941,23902,3,0,0


In [0]:
# Repetições na base de provisões

df_repeticoes_provisoes = (
    df_susep_provisoes_silver
    .groupBy(
        "ano_mes",
        "codigo_fip",
        "valor_provisao_informado"
    )
    .count()
    .filter(F.col("count") > 1)
)

# Repetições na base populacional

colunas_conteudo_quantidades = [
    "ano_mes",
    "codigo_fip",
    "grupo_populacional",
    "tipo_produto",
    "estoque_informado",
    "inclusoes_informado",
    "exclusoes_informado",
    "cancelados_informado"
]

df_repeticoes_quantidades = (
    df_susep_quantidades_silver
    .groupBy(*colunas_conteudo_quantidades)
    .count()
    .filter(F.col("count") > 1)
)

print("Repetições em provisões:")

display(
    df_repeticoes_provisoes.agg(
        F.count("*").alias("grupos_repetidos"),
        F.sum(F.col("count") - 1).alias("linhas_excedentes"),
        F.max("count").alias("maior_repeticao")
    )
)

printigeria = "Repetições em quantidades:"
print(printigeria)

display(
    df_repeticoes_quantidades.agg(
        F.count("*").alias("grupos_repetidos"),
        F.sum(F.col("count") - 1).alias("linhas_excedentes"),
        F.max("count").alias("maior_repeticao")
    )
)

Repetições em provisões:


grupos_repetidos,linhas_excedentes,maior_repeticao
0,null,null


Repetições em quantidades:


grupos_repetidos,linhas_excedentes,maior_repeticao
31660,87588,24


Nas bases de participantes e beneficiários foram encontrados grupos com conteúdo idêntico. Como os arquivos públicos não apresentam identificador de plano, contrato ou pessoa que permita distinguir a granularidade original, nenhuma linha foi excluída. Foram criadas colunas para registrar a quantidade de ocorrências e sinalizar conteúdos repetidos.

In [0]:
# Sinalização das repetições na base populacional

janela_repeticao_quantidades = Window.partitionBy(
    *colunas_conteudo_quantidades
)

df_susep_quantidades_silver = (
    df_susep_quantidades_silver
    .withColumn(
        "quantidade_ocorrencias_fonte",
        F.count("*").over(janela_repeticao_quantidades)
    )
    .withColumn(
        "flag_conteudo_repetido",
        F.col("quantidade_ocorrencias_fonte") > 1
    )
)

In [0]:
# Gravação das tabelas de provisões e quantidades

(
    df_susep_provisoes_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.susep_provisoes")
)

(
    df_susep_quantidades_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.susep_quantidades")
)

print("Tabelas SUSEP de provisões e quantidades gravadas.")

Tabelas SUSEP de provisões e quantidades gravadas.


In [0]:
#validação final da gravação
df_provisoes_gravado = spark.table(
    "workspace.silver.susep_provisoes"
)

df_quantidades_gravado = spark.table(
    "workspace.silver.susep_quantidades"
)

display(
    spark.createDataFrame(
        [
            (
                "susep_provisoes",
                df_provisoes_gravado.count()
            ),
            (
                "susep_quantidades",
                df_quantidades_gravado.count()
            )
        ],
        ["tabela", "quantidade_registros"]
    )
)

tabela,quantidade_registros
susep_provisoes,8005
susep_quantidades,270995


In [0]:
#validação final da gravação
display(
    df_quantidades_gravado.agg(
        F.count("*").alias("registros"),
        F.sum(
            F.when(F.col("flag_estoque_negativo"), 1).otherwise(0)
        ).alias("estoques_negativos_tratados"),
        F.sum(
            F.when(
                F.col("flag_movimentacao_negativa"), 1
            ).otherwise(0)
        ).alias("movimentacoes_negativas_preservadas"),
        F.sum(
            F.when(F.col("flag_conteudo_repetido"), 1).otherwise(0)
        ).alias("registros_em_grupos_repetidos"),
        F.max("quantidade_ocorrencias_fonte")
            .alias("maior_repeticao"),
        F.sum(
            F.when(F.col("estoque") < 0, 1).otherwise(0)
        ).alias("estoques_negativos_analiticos")
    )
)

registros,estoques_negativos_tratados,movimentacoes_negativas_preservadas,registros_em_grupos_repetidos,maior_repeticao,estoques_negativos_analiticos
270995,33206,12,119248,24,0


## 4. IBGE — renda e população da Região Sudeste

As tabelas do IBGE fornecem os dados demográficos e de rendimento necessários para a contextualização socioeconômica e para a construção posterior dos cenários de projeção.

Como os arquivos de origem apresentam estrutura horizontal, com os períodos distribuídos em várias colunas, nesta etapa os dados serão convertidos para o formato longitudinal, mantendo uma observação por unidade geográfica, categoria e período.

### 4.1 Inspeção inicial das tabelas Bronze do IBGE

Nesta etapa, as tabelas de rendimento e população são carregadas da camada Bronze. São verificados seus esquemas e suas quantidades de registros para identificar a estrutura dos arquivos e estabelecer valores de referência para as validações posteriores.

In [0]:
#inspecionando as duas tabelas Bronze
# Carrega a tabela Bronze com os dados de rendimento do IBGE
# referentes aos quatro estados da Região Sudeste.
df_ibge_rendimento_bronze = spark.table(
    "workspace.bronze.ibge_rendimento_7444_sudeste"
)

# Carrega a tabela Bronze com os dados de população do IBGE
# referentes aos quatro estados da Região Sudeste.
df_ibge_populacao_bronze = spark.table(
    "workspace.bronze.ibge_populacao_6407_sudeste"
)

# Exibe a estrutura da tabela de rendimento:
# nomes das colunas, tipos de dados e possibilidade de valores nulos.
print("Schema — rendimento:")
df_ibge_rendimento_bronze.printSchema()

# Exibe a estrutura da tabela de população.
print("\nSchema — população:")
df_ibge_populacao_bronze.printSchema()

# Conta os registros existentes em cada tabela Bronze.
# Esta contagem será utilizada posteriormente para validar
# se o tratamento provocou perda indevida de informações.
print(
    "\nQuantidade de registros:",
    {
        "rendimento": df_ibge_rendimento_bronze.count(),
        "populacao": df_ibge_populacao_bronze.count(),
    }
)

Schema — rendimento:
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)
 |-- _c12: string (nullable = true)
 |-- _c13: string (nullable = true)
 |-- _c14: string (nullable = true)
 |-- _c15: string (nullable = true)
 |-- _c16: string (nullable = true)
 |-- _c17: string (nullable = true)
 |-- _c18: string (nullable = true)
 |-- _c19: string (nullable = true)
 |-- _c20: string (nullable = true)
 |-- _c21: string (nullable = true)
 |-- _c22: string (nullable = true)
 |-- _c23: string (nullable = true)
 |-- _c24: string (nullable = true)
 |-- _c25: string (nullable = true)
 |-- _c26: string (nullable = true)
 |-- _c27: s

#### Inspeção dos dados de rendimento

A tabela é visualizada para identificar como os estados, os anos e os valores de rendimento estão organizados no arquivo original.

In [0]:
# Exibe os registros da tabela Bronze de rendimento.
# A visualização permite identificar a posição das unidades
# geográficas, dos períodos e dos valores antes da transformação.
display(df_ibge_rendimento_bronze)

_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9,_c10,_c11,_c12,_c13,_c14,_c15,_c16,_c17,_c18,_c19,_c20,_c21,_c22,_c23,_c24,_c25,_c26,_c27,_c28,_c29,_c30,_c31,_c32,_c33,_c34,_c35,_c36,_c37,_c38,_c39,_c40,_c41,_c42,_c43,_c44,_c45,_c46,_c47,_c48,_c49,_c50,_c51,_c52,_c53,_c54,_c55,_c56,_c57,_c58,_c59,_c60,_c61,_c62,_c63,_c64,_c65,_c66,_c67,_c68,_c69,_c70,_c71,_c72,_c73,_c74,_c75,_c76,_c77,_c78,_c79,_c80,_c81,_c82,_c83,_c84,_c85,_c86,_arquivo_origem,_data_ingestao
"Tabela 7444 - Rendimento médio mensal real das pessoas de 14 anos ou mais de idade ocupadas na semana de referência com rendimento de trabalho, de todos os trabalhos, a preços médios do último ano, por sexo",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,ibge_rendimento_7444_sudeste.csv,2026-09-08T00:41:28.677Z
"Variável - Rendimento médio mensal real das pessoas de 14 anos ou mais de idade ocupadas na semana de referência com rendimento de trabalho, habitualmente recebido em todos os trabalhos, a preços médios do último ano",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,ibge_rendimento_7444_sudeste.csv,2026-09-08T00:41:28.677Z
Nível,Cód.,Unidade da Federação,Ano x Sexo,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,ibge_rendimento_7444_sudeste.csv,2026-09-08T00:41:28.677Z
Nível,Cód.,Unidade da Federação,2012,null,null,null,null,null,2013,null,null,null,null,null,2014,null,null,null,null,null,2015,null,null,null,null,null,2016,null,null,null,null,null,2017,null,null,null,null,null,2018,null,null,null,null,null,2019,null,null,null,null,null,2020,null,null,null,null,null,2021,null,null,null,null,null,2022,null,null,null,null,null,2023,null,null,null,null,null,2024,null,null,null,null,null,2025,null,null,null,null,null,ibge_rendimento_7444_sudeste.csv,2026-09-08T00:41:28.677Z
Nível,Cód.,Unidade da Federação,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,ibge_rendimento_7444_sudeste.csv,2026-09-08T00:41:28.677Z
UF,31,Minas Gerais,2739,Reais,3124,Reais,2190,Reais,2816,Reais,3237,Reais,2217,Reais,2928,Reais,3364,Reais,2316,Reais,2842,Reais,3220,Reais,2335,Reais,2729,Reais,3062,Reais,2274,Reais,2785,Reais,3140,Reais,2299,Reais,2809,Reais,3146,Reais,2355,Reais,2788,Reais,3212,Reais,2221,Reais,2804,Reais,3139,Reais,2323,Reais,2727,Reais,2999,Reais,2347,Reais,2694,Reais,3049,Reais,2204,Reais,3105,Reais,3390,Reais,2718,Reais,3042,Reais,3374,Reais,2603,Reais,3350,Reais,3777,Reais,2790,Reais,ibge_rendimento_7444_sude

#### Inspeção dos dados populacionais

A tabela é visualizada para identificar a organização das unidades da Federação, faixas etárias, categorias de sexo, períodos e valores populacionais.

In [0]:
# Exibe os registros da tabela Bronze de população.
# A visualização permite identificar as categorias de idade,
# sexo, unidade da Federação, períodos e valores populacionais.
display(df_ibge_populacao_bronze)

_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9,_c10,_c11,_c12,_c13,_c14,_c15,_c16,_c17,_c18,_c19,_c20,_c21,_c22,_c23,_c24,_c25,_c26,_c27,_c28,_c29,_c30,_c31,_c32,_c33,_c34,_c35,_c36,_c37,_c38,_c39,_c40,_c41,_c42,_c43,_c44,_c45,_c46,_c47,_c48,_c49,_c50,_c51,_c52,_c53,_c54,_c55,_c56,_c57,_c58,_c59,_c60,_c61,_c62,_c63,_c64,_c65,_c66,_c67,_c68,_c69,_c70,_c71,_c72,_c73,_c74,_c75,_c76,_c77,_c78,_c79,_c80,_c81,_c82,_c83,_c84,_c85,_c86,_c87,_arquivo_origem,_data_ingestao
"Tabela 6407 - População residente, por sexo e grupos de idade",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,ibge_populacao_6407_sudeste.csv,2026-09-08T00:41:44.409Z
Variável - População,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,ibge_populacao_6407_sudeste.csv,2026-09-08T00:41:44.409Z
Nível,Cód.,Unidade da Federação,Grupo de idade,Ano x Sexo,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,ibge_populacao_6407_sudeste.csv,2026-09-08T00:41:44.409Z
Nível,Cód.,Unidade da Federação,Grupo de idade,2012,null,null,null,null,null,2013,null,null,null,null,null,2014,null,null,null,null,null,2015,null,null,null,null,null,2016,null,null,null,null,null,2017,null,null,null,null,null,2018,null,null,null,null,null,2019,null,null,null,null,null,2020,null,null,null,null,null,2021,null,null,null,null,null,2022,null,null,null,null,null,2023,null,null,null,null,null,2024,null,null,null,null,null,2025,null,null,null,null,null,ibge_populacao_6407_sudeste.csv,2026-09-08T00:41:44.409Z
Nível,Cód.,Unidade da Federação,Grupo de idade,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,Total,null,Homens,null,Mulheres,null,ibge_populacao_6407_sudeste.csv,2026-09-08T00:41:44.409Z
UF,31,Minas Gerais,Total,20084,Mil pessoas,9893,Mil pessoas,10191,Mil pessoas,20204,Mil pessoas,9964,Mil pessoas,10240,Mil pessoas,20330,Mil pessoas,9981,Mil pessoas,10349,Mil pessoas,20459,Mil pessoas,10024,Mil pessoas,10435,Mil pessoas,20577,Mil pessoas,10071,Mil pessoas,10506,Mil pessoas,20684,Mil pessoas,10179,Mil pessoas,10505,Mil pessoas,20790,Mil pessoas,10185,Mil pessoas,10605,Mil pessoas,20905,Mil pessoas,10360,Mil pessoas,10544,Mil pessoas,21020,Mil pessoas,10435,Mil pessoas,10585,Mil pessoas,21105,Mil pessoas,10379,Mil pessoas,10726,Mil pessoas,21166,Mil pessoas,10546,Mil pessoas,10620,Mil pessoas,21235,Mil pessoas,10521,Mil pessoas,10713,Mil pessoas,21310,Mil pessoas,10462,Mil pessoas,10848,Mil pessoas,21381,Mil pessoas,10579,Mil pessoas,10802,Mil pessoas,ibge_populacao_6

### 4.2 Tratamento dos dados de rendimento

A tabela 7444 do IBGE apresenta os anos e as categorias de sexo distribuídos horizontalmente em várias colunas. Nesta etapa, somente as linhas referentes às unidades da Federação são selecionadas e os dados são convertidos para o formato longitudinal.

O resultado terá uma linha para cada combinação de unidade da Federação, ano e sexo, facilitando consultas, agregações e cruzamentos posteriores. Os valores representam o rendimento médio mensal real, em reais.

In [0]:
from pyspark.sql import functions as F

# Define os anos disponíveis no arquivo do IBGE.
anos_ibge = list(range(2012, 2026))

# Define as categorias de sexo e suas respectivas posições
# dentro de cada conjunto de seis colunas do arquivo original.
categorias_rendimento = [
    ("TOTAL", 0),
    ("HOMENS", 2),
    ("MULHERES", 4)
]

# Monta uma lista de estruturas contendo ano, sexo, valor e unidade.
# O primeiro conjunto começa na coluna _c3 e cada novo ano
# inicia seis colunas depois.
registros_rendimento = []

for indice_ano, ano in enumerate(anos_ibge):
    coluna_inicial = 3 + (indice_ano * 6)

    for sexo, deslocamento in categorias_rendimento:
        coluna_valor = f"_c{coluna_inicial + deslocamento}"
        coluna_unidade = f"_c{coluna_inicial + deslocamento + 1}"

        registros_rendimento.append(
            F.struct(
                F.lit(ano).alias("ano"),
                F.lit(sexo).alias("sexo"),
                F.col(coluna_valor).alias("rendimento_original"),
                F.col(coluna_unidade).alias("unidade_original")
            )
        )

# Seleciona somente as linhas que representam unidades da Federação.
# As linhas de títulos, cabeçalhos, notas, fontes e legendas são descartadas.
df_ibge_rendimento_silver = (
    df_ibge_rendimento_bronze
    .filter(F.trim(F.col("_c0")) == "UF")
    .select(
        F.trim(F.col("_c1")).alias("codigo_uf"),
        F.trim(F.col("_c2")).alias("uf"),
        F.explode(F.array(*registros_rendimento)).alias("registro")
    )
    .select(
        "codigo_uf",
        "uf",
        F.col("registro.ano").cast("int").alias("ano"),
        F.col("registro.sexo").alias("sexo"),
        F.col("registro.rendimento_original").alias(
            "rendimento_original"
        ),
        F.col("registro.unidade_original").alias("unidade_original")
    )

    # Converte o rendimento para tipo numérico.
    # O valor original é mantido para fins de rastreabilidade.
    .withColumn(
        "rendimento_medio_mensal",
        F.regexp_replace(
            F.trim(F.col("rendimento_original")),
            ",",
            "."
        ).cast("decimal(18,2)")
    )

    # Padroniza a unidade de medida apresentada pelo IBGE.
    .withColumn(
        "unidade",
        F.when(
            F.upper(F.trim(F.col("unidade_original"))) == "REAIS",
            F.lit("R$")
        ).otherwise(F.trim(F.col("unidade_original")))
    )

    # Identifica valores que estavam preenchidos na fonte,
    # mas não puderam ser convertidos para o formato numérico.
    .withColumn(
        "flag_valor_nao_convertido",
        F.col("rendimento_original").isNotNull()
        & (F.trim(F.col("rendimento_original")) != "")
        & F.col("rendimento_medio_mensal").isNull()
    )

    # Registra a origem e a data de processamento do dado.
    .withColumn("fonte", F.lit("IBGE - SIDRA - Tabela 7444"))
    .withColumn("data_processamento", F.current_timestamp())

    # Organiza o resultado por estado, ano e categoria de sexo.
    .orderBy("codigo_uf", "ano", "sexo")
)

display(df_ibge_rendimento_silver)

codigo_uf,uf,ano,sexo,rendimento_original,unidade_original,rendimento_medio_mensal,unidade,flag_valor_nao_convertido,fonte,data_processamento
31,Minas Gerais,2012,HOMENS,3124,Reais,3124.00,R$,false,IBGE - SIDRA - Tabela 7444,2026-09-11T02:09:03.207Z
31,Minas Gerais,2012,MULHERES,2190,Reais,2190.00,R$,false,IBGE - SIDRA - Tabela 7444,2026-09-11T02:09:03.207Z
31,Minas Gerais,2012,TOTAL,2739,Reais,2739.00,R$,false,IBGE - SIDRA - Tabela 7444,2026-09-11T02:09:03.207Z
31,Minas Gerais,2013,HOMENS,3237,Reais,3237.00,R$,false,IBGE - SIDRA - Tabela 7444,2026-09-11T02:09:03.207Z
31,Minas Gerais,2013,MULHERES,2217,Reais,2217.00,R$,false,IBGE - SIDRA - Tabela 7444,2026-09-11T02:09:03.207Z
31,Minas Gerais,2013,TOTAL,2816,Reais,2816.00,R$,false,IBGE - SIDRA - Tabela 7444,2026-09-11T02:09:03.207Z
31,Minas Gerais,2014,HOMENS,3364,Reais,3364.00,R$,false,IBGE - SIDRA - Tabela 7444,2026-09-11T02:09:03.207Z
31,Minas Gerais,2014,MULHERES,2316,Reais,2316.00,R$,false,IBGE - SIDRA - Tabela 7444,2026-09-11T02:09:03.207Z
31,Minas Gerais,2014,TOTAL,2928,Reais,2928.00,R$,false,IBGE - SIDRA - Tabela 7444,2026-09-11T02:09:03.207Z
31,Minas Gerais,2015,HOMENS,3220,Reais,3220.00,R$,false,IBGE - SIDRA - Tabela 7444,2026-09-11T02:09:03.207Z


### 4.3 Validação dos dados de rendimento

Após a transformação para o formato longitudinal, são verificadas a quantidade de registros, a cobertura geográfica e temporal, as categorias de sexo, possíveis falhas de conversão, valores nulos, valores negativos e duplicidades.

Considerando quatro unidades da Federação, três categorias de sexo e quatorze anos, são esperados 168 registros.

In [0]:
# Valida a quantidade de registros, a cobertura dos dados
# e a qualidade da conversão dos valores de rendimento.
df_validacao_rendimento = (
    df_ibge_rendimento_silver
    .agg(
        # Quantidade total de registros após a transformação.
        F.count("*").alias("registros"),

        # Quantidade de estados distintos presentes na tabela.
        F.countDistinct("codigo_uf").alias("ufs_distintas"),

        # Primeiro e último ano disponíveis.
        F.min("ano").alias("primeiro_ano"),
        F.max("ano").alias("ultimo_ano"),

        # Quantidade de categorias de sexo.
        F.countDistinct("sexo").alias("categorias_sexo"),

        # Valores preenchidos na origem que não puderam
        # ser convertidos para o tipo numérico.
        F.sum(
            F.when(F.col("flag_valor_nao_convertido"), 1).otherwise(0)
        ).alias("valores_nao_convertidos"),

        # Valores numéricos ausentes após a transformação.
        F.sum(
            F.when(F.col("rendimento_medio_mensal").isNull(), 1).otherwise(0)
        ).alias("valores_nulos"),

        # Valores negativos, caso existam.
        F.sum(
            F.when(F.col("rendimento_medio_mensal") < 0, 1).otherwise(0)
        ).alias("valores_negativos")
    )
)

display(df_validacao_rendimento)

registros,ufs_distintas,primeiro_ano,ultimo_ano,categorias_sexo,valores_nao_convertidos,valores_nulos,valores_negativos
168,4,2012,2025,3,0,0,0


In [0]:
# Verifica se existe mais de um registro para a mesma combinação
# de unidade da Federação, ano e categoria de sexo.
df_repeticoes_rendimento = (
    df_ibge_rendimento_silver
    .groupBy("codigo_uf", "ano", "sexo")
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.count("*").alias("grupos_repetidos"),
        F.sum(F.col("count") - 1).alias("linhas_excedentes"),
        F.max("count").alias("maior_repeticao")
    )
)

display(df_repeticoes_rendimento)

grupos_repetidos,linhas_excedentes,maior_repeticao
0,null,null


### 4.4 Gravação da tabela de rendimento na camada Silver

Após a aprovação das validações de estrutura e qualidade, os dados tratados de rendimento são armazenados em formato Delta na camada Silver.

A gravação utiliza o modo de sobrescrita, permitindo que a tabela seja recriada de maneira controlada caso o processamento seja executado novamente.

In [0]:
# Define o nome completo da tabela de destino na camada Silver.
tabela_ibge_rendimento_silver = (
    "workspace.silver.ibge_rendimento_7444_sudeste"
)

# Grava os dados tratados em formato Delta.
# O modo overwrite substitui uma eventual versão anterior da tabela.
(
    df_ibge_rendimento_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_ibge_rendimento_silver)
)

print(
    f"Tabela gravada com sucesso: "
    f"{tabela_ibge_rendimento_silver}"
)

Tabela gravada com sucesso: workspace.silver.ibge_rendimento_7444_sudeste


In [0]:
# Lê novamente a tabela diretamente da camada Silver.
# Essa leitura confirma que a gravação foi concluída
# e que os dados podem ser recuperados do catálogo.
df_ibge_rendimento_gravado = spark.table(
    "workspace.silver.ibge_rendimento_7444_sudeste"
)

# Confere a quantidade de registros e os limites temporais
# existentes na tabela efetivamente armazenada.
df_validacao_rendimento_gravado = (
    df_ibge_rendimento_gravado
    .agg(
        F.count("*").alias("quantidade_registros"),
        F.countDistinct("codigo_uf").alias("ufs_distintas"),
        F.min("ano").alias("primeiro_ano"),
        F.max("ano").alias("ultimo_ano"),
        F.countDistinct("sexo").alias("categorias_sexo"),
        F.sum(
            F.when(F.col("flag_valor_nao_convertido"), 1).otherwise(0)
        ).alias("valores_nao_convertidos")
    )
)

display(df_validacao_rendimento_gravado)

quantidade_registros,ufs_distintas,primeiro_ano,ultimo_ano,categorias_sexo,valores_nao_convertidos
168,4,2012,2025,3,0


### 4.5 Tratamento dos dados populacionais

A tabela 6407 do IBGE apresenta a população residente por unidade da Federação, grupo de idade, ano e sexo. Assim como a tabela de rendimento, os períodos e as categorias de sexo estão distribuídos horizontalmente.

Nesta etapa, os dados são convertidos para o formato longitudinal. Os valores originais, expressos em milhares de pessoas, são preservados, e também é calculada uma estimativa da população em número de pessoas.

In [0]:
# Define os anos encontrados no arquivo populacional do IBGE.
anos_ibge = list(range(2012, 2026))

# Define as categorias de sexo e suas posições
# dentro de cada conjunto de seis colunas.
categorias_populacao = [
    ("TOTAL", 0),
    ("HOMENS", 2),
    ("MULHERES", 4)
]

# Monta as estruturas que serão utilizadas para transformar
# as colunas dos anos e sexos em linhas.
registros_populacao = []

for indice_ano, ano in enumerate(anos_ibge):
    # Na tabela de população, o primeiro valor está na coluna _c4.
    # Cada ano seguinte começa seis colunas depois.
    coluna_inicial = 4 + (indice_ano * 6)

    for sexo, deslocamento in categorias_populacao:
        coluna_valor = f"_c{coluna_inicial + deslocamento}"
        coluna_unidade = f"_c{coluna_inicial + deslocamento + 1}"

        registros_populacao.append(
            F.struct(
                F.lit(ano).alias("ano"),
                F.lit(sexo).alias("sexo"),
                F.col(coluna_valor).alias("populacao_original"),
                F.col(coluna_unidade).alias("unidade_original")
            )
        )

# Seleciona somente as linhas referentes às unidades da Federação
# e transforma os dados do formato horizontal para o longitudinal.
df_ibge_populacao_silver = (
    df_ibge_populacao_bronze
    .filter(F.trim(F.col("_c0")) == "UF")
    .select(
        F.trim(F.col("_c1")).alias("codigo_uf"),
        F.trim(F.col("_c2")).alias("uf"),
        F.trim(F.col("_c3")).alias("grupo_idade"),
        F.explode(F.array(*registros_populacao)).alias("registro")
    )
    .select(
        "codigo_uf",
        "uf",
        "grupo_idade",
        F.col("registro.ano").cast("int").alias("ano"),
        F.col("registro.sexo").alias("sexo"),
        F.col("registro.populacao_original").alias(
            "populacao_original"
        ),
        F.col("registro.unidade_original").alias(
            "unidade_original"
        )
    )

    # Converte o valor original para decimal.
    # A unidade informada pelo IBGE é mil pessoas.
    .withColumn(
        "populacao_mil_pessoas",
        F.regexp_replace(
            F.trim(F.col("populacao_original")),
            ",",
            "."
        ).cast("decimal(18,3)")
    )

    # Converte o valor de milhares para uma estimativa
    # da quantidade absoluta de pessoas.
    .withColumn(
        "populacao_pessoas",
        F.round(
            F.col("populacao_mil_pessoas") * F.lit(1000)
        ).cast("long")
    )

    # Padroniza a descrição da unidade de medida.
    .withColumn(
        "unidade",
        F.when(
            F.upper(F.trim(F.col("unidade_original"))) == "MIL PESSOAS",
            F.lit("mil pessoas")
        ).otherwise(F.trim(F.col("unidade_original")))
    )

    # Identifica valores preenchidos na origem que não puderam
    # ser convertidos para número.
    .withColumn(
        "flag_valor_nao_convertido",
        F.col("populacao_original").isNotNull()
        & (F.trim(F.col("populacao_original")) != "")
        & F.col("populacao_mil_pessoas").isNull()
    )

    # Acrescenta informações de rastreabilidade.
    .withColumn("fonte", F.lit("IBGE - SIDRA - Tabela 6407"))
    .withColumn("data_processamento", F.current_timestamp())

    # Organiza o resultado para facilitar a conferência.
    .orderBy("codigo_uf", "grupo_idade", "ano", "sexo")
)

display(df_ibge_populacao_silver)

codigo_uf,uf,grupo_idade,ano,sexo,populacao_original,unidade_original,populacao_mil_pessoas,populacao_pessoas,unidade,flag_valor_nao_convertido,fonte,data_processamento
31,Minas Gerais,18 a 19 anos,2012,HOMENS,326,Mil pessoas,326.000,326000,mil pessoas,false,IBGE - SIDRA - Tabela 6407,2026-09-11T02:16:31.475Z
31,Minas Gerais,18 a 19 anos,2012,MULHERES,311,Mil pessoas,311.000,311000,mil pessoas,false,IBGE - SIDRA - Tabela 6407,2026-09-11T02:16:31.475Z
31,Minas Gerais,18 a 19 anos,2012,TOTAL,637,Mil pessoas,637.000,637000,mil pessoas,false,IBGE - SIDRA - Tabela 6407,2026-09-11T02:16:31.475Z
31,Minas Gerais,18 a 19 anos,2013,HOMENS,343,Mil pessoas,343.000,343000,mil pessoas,false,IBGE - SIDRA - Tabela 6407,2026-09-11T02:16:31.475Z
31,Minas Gerais,18 a 19 anos,2013,MULHERES,349,Mil pessoas,349.000,349000,mil pessoas,false,IBGE - SIDRA - Tabela 6407,2026-09-11T02:16:31.475Z
31,Minas Gerais,18 a 19 anos,2013,TOTAL,692,Mil pessoas,692.000,692000,mil pessoas,false,IBGE - SIDRA - Tabela 6407,2026-09-11T02:16:31.475Z
31,Minas Gerais,18 a 19 anos,2014,HOMENS,351,Mil pessoas,351.000,351000,mil pessoas,false,IBGE - SIDRA - Tabela 6407,2026-09-11T02:16:31.475Z
31,Minas Gerais,18 a 19 anos,2014,MULHERES,305,Mil pessoas,305.000,305000,mil pessoas,false,IBGE - SIDRA - Tabela 6407,2026-09-11T02:16:31.475Z
31,Minas Gerais,18 a 19 anos,2014,TOTAL,656,Mil pessoas,656.000,656000,mil pessoas,false,IBGE - SIDRA - Tabela 6407,2026-09-11T02:16:31.475Z
31,Minas Gerais,18 a 19 anos,2015,HOMENS,304,Mil pessoas,304.000,304000,mil pessoas,false,IBGE - SIDRA - Tabela 6407,2026-09-11T02:16:31.475Z


### 4.6 Validação dos dados populacionais

Após a conversão para o formato longitudinal, são verificadas a quantidade de registros, a cobertura geográfica e temporal, os grupos de idade, as categorias de sexo, possíveis falhas de conversão, valores nulos, negativos e duplicidades.

A consistência entre os valores de homens, mulheres e total também é avaliada. Pequenas diferenças podem ocorrer devido ao arredondamento dos valores originais, que são divulgados em milhares de pessoas.

In [0]:
# Calcula os principais indicadores de estrutura e qualidade
# da tabela populacional transformada.
df_validacao_populacao = (
    df_ibge_populacao_silver
    .agg(
        # Quantidade total de registros.
        F.count("*").alias("registros"),

        # Quantidade de estados e grupos etários distintos.
        F.countDistinct("codigo_uf").alias("ufs_distintas"),
        F.countDistinct("grupo_idade").alias("grupos_idade"),

        # Primeiro e último ano disponíveis.
        F.min("ano").alias("primeiro_ano"),
        F.max("ano").alias("ultimo_ano"),

        # Quantidade de categorias de sexo.
        F.countDistinct("sexo").alias("categorias_sexo"),

        # Valores preenchidos na fonte que não puderam
        # ser convertidos para o tipo numérico.
        F.sum(
            F.when(F.col("flag_valor_nao_convertido"), 1).otherwise(0)
        ).alias("valores_nao_convertidos"),

        # Valores populacionais nulos após a conversão.
        F.sum(
            F.when(F.col("populacao_mil_pessoas").isNull(), 1).otherwise(0)
        ).alias("valores_nulos"),

        # Valores populacionais negativos.
        F.sum(
            F.when(F.col("populacao_mil_pessoas") < 0, 1).otherwise(0)
        ).alias("valores_negativos")
    )
)

display(df_validacao_populacao)

registros,ufs_distintas,grupos_idade,primeiro_ano,ultimo_ano,categorias_sexo,valores_nao_convertidos,valores_nulos,valores_negativos
1512,4,9,2012,2025,3,0,0,0


In [0]:
# Verifica se existe mais de um registro para a mesma combinação
# de UF, grupo de idade, ano e categoria de sexo.
df_repeticoes_populacao = (
    df_ibge_populacao_silver
    .groupBy("codigo_uf", "grupo_idade", "ano", "sexo")
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.count("*").alias("grupos_repetidos"),
        F.sum(F.col("count") - 1).alias("linhas_excedentes"),
        F.max("count").alias("maior_repeticao")
    )
)

display(df_repeticoes_populacao)

grupos_repetidos,linhas_excedentes,maior_repeticao
0,null,null


In [0]:
# Coloca as categorias de sexo em colunas para comparar
# o total informado com a soma de homens e mulheres.
df_consistencia_sexo_populacao = (
    df_ibge_populacao_silver
    .groupBy("codigo_uf", "uf", "grupo_idade", "ano")
    .pivot("sexo", ["TOTAL", "HOMENS", "MULHERES"])
    .agg(F.first("populacao_mil_pessoas"))

    # Calcula a diferença entre o total informado
    # e a soma das populações masculina e feminina.
    .withColumn(
        "diferenca_mil_pessoas",
        F.abs(
            F.col("TOTAL")
            - (F.col("HOMENS") + F.col("MULHERES"))
        )
    )
)

# Resume as diferenças encontradas.
# Diferenças pequenas são esperadas devido ao arredondamento
# dos valores divulgados em milhares de pessoas.
df_validacao_consistencia_sexo = (
    df_consistencia_sexo_populacao
    .agg(
        F.count("*").alias("grupos_avaliados"),
        F.max("diferenca_mil_pessoas").alias("maior_diferenca"),
        F.sum(
            F.when(F.col("diferenca_mil_pessoas") > 1, 1).otherwise(0)
        ).alias("grupos_com_diferenca_superior_a_1_mil")
    )
)

display(df_validacao_consistencia_sexo)

grupos_avaliados,maior_diferenca,grupos_com_diferenca_superior_a_1_mil
504,1.000,0


### 4.7 Gravação da tabela populacional na camada Silver

A tabela populacional somente será gravada após a confirmação automática dos critérios de qualidade. As verificações impedem a sobrescrita da tabela de destino caso sejam encontrados registros ausentes, falhas de conversão, valores negativos ou uma estrutura diferente da esperada.

In [0]:
# Consolida os resultados das validações em um único registro.
resultado_populacao = (
    df_ibge_populacao_silver
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("codigo_uf").alias("ufs_distintas"),
        F.countDistinct("grupo_idade").alias("grupos_idade"),
        F.min("ano").alias("primeiro_ano"),
        F.max("ano").alias("ultimo_ano"),
        F.countDistinct("sexo").alias("categorias_sexo"),
        F.sum(
            F.when(F.col("flag_valor_nao_convertido"), 1).otherwise(0)
        ).alias("valores_nao_convertidos"),
        F.sum(
            F.when(F.col("populacao_mil_pessoas").isNull(), 1).otherwise(0)
        ).alias("valores_nulos"),
        F.sum(
            F.when(F.col("populacao_mil_pessoas") < 0, 1).otherwise(0)
        ).alias("valores_negativos")
    )
    .first()
)

# Interrompe a execução antes da gravação caso algum
# critério obrigatório de qualidade não seja atendido.
assert resultado_populacao["registros"] == 1512, \
    "Quantidade inesperada de registros."

assert resultado_populacao["ufs_distintas"] == 4, \
    "Quantidade inesperada de UFs."

assert resultado_populacao["grupos_idade"] == 9, \
    "Quantidade inesperada de grupos de idade."

assert resultado_populacao["primeiro_ano"] == 2012, \
    "Primeiro ano inesperado."

assert resultado_populacao["ultimo_ano"] == 2025, \
    "Último ano inesperado."

assert resultado_populacao["categorias_sexo"] == 3, \
    "Quantidade inesperada de categorias de sexo."

assert resultado_populacao["valores_nao_convertidos"] == 0, \
    "Foram encontradas falhas de conversão."

assert resultado_populacao["valores_nulos"] == 0, \
    "Foram encontrados valores populacionais nulos."

assert resultado_populacao["valores_negativos"] == 0, \
    "Foram encontrados valores populacionais negativos."

# A gravação somente será alcançada se todas as verificações
# anteriores forem concluídas sem erro.
tabela_ibge_populacao_silver = (
    "workspace.silver.ibge_populacao_6407_sudeste"
)

(
    df_ibge_populacao_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_ibge_populacao_silver)
)

print(
    "Validações aprovadas e tabela gravada com sucesso:",
    tabela_ibge_populacao_silver
)

Validações aprovadas e tabela gravada com sucesso: workspace.silver.ibge_populacao_6407_sudeste


In [0]:
# Lê novamente os dados diretamente da tabela Silver.
df_ibge_populacao_gravado = spark.table(
    "workspace.silver.ibge_populacao_6407_sudeste"
)

# Confirma os principais indicadores após a gravação.
df_validacao_populacao_gravado = (
    df_ibge_populacao_gravado
    .agg(
        F.count("*").alias("quantidade_registros"),
        F.countDistinct("codigo_uf").alias("ufs_distintas"),
        F.countDistinct("grupo_idade").alias("grupos_idade"),
        F.min("ano").alias("primeiro_ano"),
        F.max("ano").alias("ultimo_ano"),
        F.countDistinct("sexo").alias("categorias_sexo")
    )
)

display(df_validacao_populacao_gravado)

quantidade_registros,ufs_distintas,grupos_idade,primeiro_ano,ultimo_ano,categorias_sexo
1512,4,9,2012,2025,3


## 5. Validação final da camada Silver

Ao final do processamento, todas as tabelas tratadas são consultadas diretamente no catálogo do Databricks. Essa verificação confirma que os dados foram efetivamente armazenados na camada Silver e registra a quantidade de linhas disponível em cada tabela.

A camada Silver contém dados padronizados, tipados e acompanhados por indicadores de qualidade e rastreabilidade. Valores potencialmente anômalos foram preservados nos campos originais e, quando necessário, separados dos campos destinados às análises.

In [0]:
# Relação das tabelas produzidas no notebook Silver.
tabelas_silver = [
    "workspace.silver.previc_dsi_2025",
    "workspace.silver.previc_epb_2025",
    "workspace.silver.susep_lista_empresas",
    "workspace.silver.susep_contrib_benef",
    "workspace.silver.susep_fundos",
    "workspace.silver.susep_resgates",
    "workspace.silver.susep_previdencia_uf",
    "workspace.silver.susep_provisoes",
    "workspace.silver.susep_quantidades",
    "workspace.silver.ibge_rendimento_7444_sudeste",
    "workspace.silver.ibge_populacao_6407_sudeste"
]

# Consulta diretamente cada tabela gravada e conta seus registros.
# Essa leitura final comprova que as tabelas existem no catálogo
# e podem ser utilizadas na próxima camada do projeto.
contagens_silver = []

for nome_tabela in tabelas_silver:
    quantidade = spark.table(nome_tabela).count()

    contagens_silver.append(
        (nome_tabela, quantidade)
    )

# Cria um DataFrame com o inventário final da camada Silver.
df_inventario_silver = spark.createDataFrame(
    contagens_silver,
    ["tabela", "quantidade_registros"]
)

display(df_inventario_silver.orderBy("tabela"))

tabela,quantidade_registros
workspace.silver.ibge_populacao_6407_sudeste,1512
workspace.silver.ibge_rendimento_7444_sudeste,168
workspace.silver.previc_dsi_2025,9938
workspace.silver.previc_epb_2025,331840
workspace.silver.susep_contrib_benef,27605
workspace.silver.susep_fundos,12992
workspace.silver.susep_lista_empresas,233
workspace.silver.susep_previdencia_uf,406034
workspace.silver.susep_provisoes,8005
workspace.silver.susep_quantidades,270995


## 6. Conclusão

A camada Silver foi concluída com a padronização das bases da PREVIC, SUSEP e IBGE.

Os tratamentos realizados incluíram:

- conversão e padronização de tipos de dados;
- normalização de datas, códigos e unidades de medida;
- consolidação de arquivos relacionados;
- transformação das tabelas horizontais do IBGE para o formato longitudinal;
- identificação de falhas de conversão;
- verificação de valores nulos e negativos;
- identificação de registros repetidos;
- preservação dos valores originais para rastreabilidade;
- criação de campos analíticos e indicadores de qualidade;
- validação das tabelas após sua gravação em formato Delta.

As tabelas resultantes estão preparadas para a construção da camada Gold, na qual serão produzidos os indicadores, cruzamentos, recortes para a Região Sudeste e cenários relacionados à reposição de renda.